# SGRPO: Unified Kaggle Pipeline
This notebook recreates the SGRPO repository structure in `/kaggle/working/` and runs a quick test.

In [ ]:
!mkdir -p data losses models rewards rollouts trainer tracking

In [ ]:
%%writefile requirements.txt
# Core ML
torch>=2.5.0
numpy>=2.0.0,<2.5.0
pandas>=3.0.3
scipy>=1.14.0
statsmodels>=0.14.0
scikit-learn>=1.5.0

# LLM & RL
transformers>=4.45.0
datasets>=3.0.0
trl>=0.12.0
accelerate>=1.0.0
vllm>=0.6.0; sys_platform != 'win32'
peft>=0.13.0
bitsandbytes>=0.45.0
huggingface-hub>=0.25.0
tokenizers>=0.20.0
sentencepiece>=0.2.0
protobuf>=5.0.0
mamba-ssm>=1.2.0; sys_platform == 'linux'
causal-conv1d>=1.2.0; sys_platform == 'linux'

# Experiment Management
wandb
omegaconf>=2.3.0
python-dotenv>=1.0.0
tqdm>=4.66.0
pyyaml>=6.0.0
joblib>=1.4.0
optuna

# Visualization
matplotlib
seaborn
plotly>=6.8.0

# System Monitoring
psutil>=6.0.0
pynvml>=12.0.0

#Tensor Operations
einops>=0.8.0

In [ ]:
%%writefile config.py
"""
config.py

Centralized configuration for the SGRPO research project.

Responsibilities:
1. Typed hyperparameter defaults in a dataclass
2. Deterministic seed propagation (torch, numpy, random, CUDA)
3. .env loading for WandB API key
4. Hardware and environment info capture for reproducibility
5. Serialization to dict for WandB config logging
6. Git commit hash capture when available

Every experiment must be reproducible by another researcher
without additional instructions. This module enforces that.
"""

import os
import sys
import random
import platform
import subprocess
from dataclasses import dataclass, field, asdict
from typing import Optional

import numpy as np
import torch
from dotenv import load_dotenv


# ── Load .env at import time ────────────────────────────────────────────────
# This ensures WANDB_API_KEY is available before any wandb import
load_dotenv()

# Map the .env key name to what wandb expects
_wandb_key = os.environ.get("wandb_api")
if _wandb_key and not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = _wandb_key


@dataclass
class TrainingConfig:
    """
    All hyperparameters for a single training run.
    Designed so that two runs with identical configs produce identical results
    (given deterministic seeds and hardware).
    """

    # ── Algorithm ────────────────────────────────────────────────────────────
    algorithm: str = "sgrpo"  # ppo | grpo | dapo | bapo | sgrpo

    # ── Model ────────────────────────────────────────────────────────────────
    model_name: str = "state-spaces/mamba-130m-hf"
    dtype: str = "bfloat16"  # bfloat16 | float32

    # ── Dataset ──────────────────────────────────────────────────────────────
    dataset: str = "gsm8k"    # any key in main.py's DATASETS registry

    # ── Training ─────────────────────────────────────────────────────────────
    steps: int = 500
    lr: float = 1e-6
    weight_decay: float = 0.0
    max_grad_norm: float = 1.0
    batch_size: int = 1       # prompts per step (keep 1 for 6GB VRAM)
    grad_accum: int = 8       # gradient accumulation steps
    warmup_steps: int = 0
    # PPO-style inner optimization epochs: reuse each rollout batch mu times.
    # mu == 1 (default) is the legacy single-pass behavior (bit-identical).
    # mu > 1 re-optimizes the same rollouts, stepping the optimizer after each
    # inner epoch so theta moves — this is what makes the importance ratio and
    # Future-KL non-trivial (with mu == 1 they are identically 1.0 / 0.0).
    # Default kept at 1 for backward compatibility; 2-4 recommended for real
    # runs (see sgrpo_research_roadmap.md Fix 3).
    inner_epochs: int = 1

    # ── RL ────────────────────────────────────────────────────────────────────
    group_size: int = 4       # G — rollouts per prompt (1 for PPO)
    clip_eps: float = 0.2
    kl_coef: float = 0.01
    temperature: float = 1.0
    max_tokens: int = 256     # max generated tokens per rollout

    # ── SGRPO-specific ───────────────────────────────────────────────────────
    future_kl_decay: float = 30.0
    future_kl_clip_low: float = 1.0
    future_kl_clip_high: float = 1.2

    # ── DAPO-specific ────────────────────────────────────────────────────────
    dapo_epsilon_high: float = 0.28

    # ── BAPO-specific (paper notation in comments) ───────────────────────────
    bapo_rho_target: float = 0.5    # rho_0 — positive token contribution threshold
    bapo_c_low_min: float = 0.8     # a- — initial (tightest) lower clipping bound
    bapo_c_low_max: float = 0.9     # b- — maximum the lower bound may be raised to
    bapo_c_high_min: float = 1.2    # a+ — initial (tightest) upper clipping bound
    bapo_c_high_max: float = 1.32   # b+ — maximum the upper bound may be widened to
    bapo_delta_high: float = 0.01   # d1 — step size of the upper bound
    bapo_delta_low: float = 0.01    # d2 — step size of the lower bound

    # ── Evaluation ───────────────────────────────────────────────────────────
    eval_every: int = 50      # evaluate every N steps
    eval_samples: int = 50    # number of test prompts per eval
    histogram_every: int = 50
    sample_table_every: int = 100
    checkpoint_every: int = 250

    # ── Reproducibility ──────────────────────────────────────────────────────
    seed: int = 42
    deterministic: bool = True  # torch.use_deterministic_algorithms

    # ── System ───────────────────────────────────────────────────────────────
    device: str = "cpu"
    # Multi-GPU / cloud training via HuggingFace Accelerate. Launch with
    # `accelerate launch main.py --use_accelerate ...`. When False (default)
    # the trainer is plain single-device PyTorch — local behavior unchanged.
    use_accelerate: bool = False

    # ── WandB ────────────────────────────────────────────────────────────────
    wandb_project: str = "rl-algo-comparison-2026"
    run_name: str = "trial-1"
    no_wandb: bool = False

    # ── Paths ────────────────────────────────────────────────────────────────
    checkpoint_dir: str = "checkpoints"
    # Include optimizer state in checkpoints (needed to resume training).
    # The benchmark disables this: its checkpoints exist only for test-set
    # evaluation, and AdamW state roughly triples the file size.
    save_optimizer_state: bool = True

    def __post_init__(self):
        """Enforce algorithm-specific constraints."""
        if self.algorithm == "ppo" and self.group_size != 1:
            print("Warning: PPO uses group_size=1. Overriding.")
            self.group_size = 1

    def to_dict(self) -> dict:
        """Serialize to dict for WandB config, including environment info."""
        d = asdict(self)
        d.update(self._get_environment_info())
        return d

    def _get_environment_info(self) -> dict:
        """Capture full environment for reproducibility."""
        info = {
            "env/python_version": sys.version,
            "env/platform": platform.platform(),
            "env/torch_version": torch.__version__,
            "env/cuda_available": torch.cuda.is_available(),
        }

        if torch.cuda.is_available():
            info["env/gpu_name"] = torch.cuda.get_device_name(0)
            info["env/gpu_memory_gb"] = round(
                torch.cuda.get_device_properties(0).total_memory / 1e9, 2
            )
            info["env/cuda_version"] = torch.version.cuda or "N/A"

        # Git commit hash
        try:
            commit = subprocess.check_output(
                ["git", "rev-parse", "HEAD"],
                stderr=subprocess.DEVNULL,
                cwd=os.path.dirname(os.path.abspath(__file__)),
            ).decode().strip()
            info["env/git_commit"] = commit
        except (subprocess.CalledProcessError, FileNotFoundError):
            info["env/git_commit"] = "unknown"

        try:
            import transformers
            info["env/transformers_version"] = transformers.__version__
        except ImportError:
            pass

        return info


def set_deterministic_seeds(seed: int, deterministic: bool = True) -> None:
    """
    Set all random seeds for full reproducibility.

    Covers:
    - Python's random module
    - NumPy's random generator
    - PyTorch CPU and CUDA generators
    - CUDA deterministic algorithms (when deterministic=True)
    - CUBLAS workspace config for determinism
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        # CUBLAS deterministic — required for full reproducibility
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except TypeError:
            # Older PyTorch versions
            torch.use_deterministic_algorithms(True)


def config_from_args(args) -> TrainingConfig:
    """Convert argparse Namespace to TrainingConfig."""
    return TrainingConfig(**{
        k: v for k, v in vars(args).items()
        if k in TrainingConfig.__dataclass_fields__
    })


In [ ]:
%%writefile main.py
"""
main.py

Entry point for the SGRPO research project.
One command runs any algorithm with identical infrastructure.

Usage:
    python main.py --algorithm ppo   --steps 500
    python main.py --algorithm grpo  --steps 500 --group_size 4
    python main.py --algorithm dapo  --steps 500 --group_size 4
    python main.py --algorithm bapo  --steps 500 --group_size 4
    python main.py --algorithm sgrpo --steps 500 --group_size 4

    # Disable W&B for local testing
    python main.py --algorithm sgrpo --steps 50 --no_wandb --device cpu

Swapping --algorithm is the ONLY change needed to run a different algorithm.
Everything else — model, dataset, trainer, logging — is identical.
This guarantees a fair comparison.
"""

import argparse
import torch

# Config module handles .env loading at import time
from config import TrainingConfig, set_deterministic_seeds, config_from_args
from models.load_model import load_model, detect_architecture
from data import DATASETS
from trainer.base_trainer import BaseTrainer


def get_args():
    parser = argparse.ArgumentParser(
        description="SGRPO Research: Custom RL Algorithm Trainer"
    )

    # ── Algorithm ────────────────────────────────────────────────────────
    parser.add_argument(
        "--algorithm", type=str, required=True,
        choices=["ppo", "grpo", "dapo", "bapo", "sgrpo"],
        help="Which algorithm to run"
    )

    # ── Model ────────────────────────────────────────────────────────────
    parser.add_argument("--model_name", type=str,
                        default="state-spaces/mamba-130m-hf",
                        help="HuggingFace model name or path. SSM/hybrid "
                             "models get state isolation under sgrpo; "
                             "transformers fall back to standard rollouts "
                             "(control condition).")

    # ── Dataset ──────────────────────────────────────────────────────────
    parser.add_argument("--dataset", type=str, default="gsm8k",
                        choices=sorted(DATASETS.keys()),
                        help="Benchmark to train/eval on. All benchmarks "
                             "use verifiable rewards (no reward model).")

    # ── Training ─────────────────────────────────────────────────────────
    parser.add_argument("--steps",      type=int,   default=500)
    parser.add_argument("--group_size", type=int,   default=4,
                        help="G — rollouts per prompt. Use 1 for PPO.")
    parser.add_argument("--lr",         type=float, default=1e-6)
    parser.add_argument("--max_tokens", type=int,   default=256)
    parser.add_argument("--batch_size", type=int,   default=1,
                        help="Prompts per step. Keep at 1 for 6GB VRAM.")
    parser.add_argument("--grad_accum", type=int,   default=8,
                        help="Gradient accumulation steps (across prompts).")
    parser.add_argument("--inner_epochs", type=int, default=1,
                        help="PPO-style inner optimization epochs: reuse each "
                             "rollout batch mu times. 1 = legacy single pass "
                             "(ratio/Future-KL trivial); 2-4 recommended for "
                             "real runs so theta moves between passes.")
    parser.add_argument("--clip_eps",   type=float, default=0.2)
    parser.add_argument("--temperature", type=float, default=1.0)

    # ── SGRPO-specific ───────────────────────────────────────────────────
    parser.add_argument("--future_kl_decay",     type=float, default=30.0)
    parser.add_argument("--future_kl_clip_low",  type=float, default=1.0)
    parser.add_argument("--future_kl_clip_high", type=float, default=1.2)

    # ── Reproducibility ──────────────────────────────────────────────────
    parser.add_argument("--seed",       type=int,   default=42)
    parser.add_argument("--device",     type=str,   default="cuda")

    # ── Multi-GPU / cloud ────────────────────────────────────────────────
    parser.add_argument("--use_accelerate", action="store_true",
                        help="Enable HuggingFace Accelerate (multi-GPU, "
                             "mixed precision). Run via: "
                             "accelerate launch main.py --use_accelerate ...")

    # ── Evaluation ───────────────────────────────────────────────────────
    parser.add_argument("--eval_every",  type=int, default=50)
    parser.add_argument("--eval_samples", type=int, default=50)

    # ── WandB ────────────────────────────────────────────────────────────
    parser.add_argument("--run_name",      type=str, default="trial-1")
    parser.add_argument("--wandb_project", type=str, default="rl-algo-comparison-2026")
    parser.add_argument("--no_wandb",      action="store_true",
                        help="Disable W&B logging (for quick local tests)")

    return parser.parse_args()


def main():
    args = get_args()
    config = config_from_args(args)

    # ── Reproducibility ──────────────────────────────────────────────────
    set_deterministic_seeds(config.seed, config.deterministic)

    print(f"\n{'='*60}")
    print(f"SGRPO Research | Algorithm: {config.algorithm.upper()}")
    print(f"Model: {config.model_name} | Dataset: {config.dataset}")
    print(f"Steps: {config.steps} | Group size: {config.group_size}")
    print(f"LR: {config.lr} | Clip eps: {config.clip_eps}")
    print(f"Seed: {config.seed} | Device: {config.device}")
    print(f"Eval every: {config.eval_every} steps")
    if config.algorithm == "sgrpo":
        print(f"Future-KL: decay={config.future_kl_decay}, "
              f"clip=[{config.future_kl_clip_low}, {config.future_kl_clip_high}]")
    print(f"{'='*60}\n")

    # ── Load model and data ──────────────────────────────────────────────
    # Under Accelerate, load on CPU and let accelerator.prepare() place the
    # model — loading straight to "cuda" would put every rank on cuda:0.
    model, tokenizer = load_model(
        device="cpu" if config.use_accelerate else config.device,
        dtype=config.dtype,
        model_name=config.model_name,
    )
    arch = detect_architecture(model)
    dataset_cls = DATASETS[config.dataset]
    train_dataset = dataset_cls(split="train")
    eval_dataset = dataset_cls(split="test")

    # ── Build trainer ────────────────────────────────────────────────────
    trainer = BaseTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        algorithm=config.algorithm,
        config=config,
        arch=arch,
    )

    trainer.train()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile data/__init__.py
"""
data/__init__.py

Benchmark registry — every entry implements BaseRewardDataset, so the
trainer and every experiment script need no changes when a new benchmark
is added here. main.py and Experiments/run_benchmark.py both read this
registry.
"""

from data.base_dataset import BaseRewardDataset
from data.gsm8k import GSM8KDataset
from data.math_dataset import MATHDataset

DATASETS: dict[str, type[BaseRewardDataset]] = {
    "gsm8k": GSM8KDataset,
    "math": MATHDataset,
}

__all__ = ["BaseRewardDataset", "GSM8KDataset", "MATHDataset", "DATASETS"]


In [ ]:
%%writefile data/base_dataset.py
"""
data/base_dataset.py

One job: define the interface every benchmark dataset must implement so the
trainer can run on ANY benchmark without modification.

Design contract (roadmap §1.3.2):
    - Every item is a dict with at least:
          "prompt":  formatted string ready for tokenization
          "answer":  ground-truth string used by compute_reward
    - compute_reward(generated_text, ground_truth) -> float must be
      VERIFIABLE (deterministic, no learned reward model). This keeps
      reward variance out of the algorithm comparison — the whole point
      of using verifiable benchmarks.

The trainer discovers the reward function via the dataset object
(`dataset.compute_group_rewards`), so adding a new benchmark means adding
one file that subclasses BaseRewardDataset — no trainer edits.
"""

from abc import ABC, abstractmethod

from torch.utils.data import Dataset


class BaseRewardDataset(Dataset, ABC):
    """Interface for any benchmark dataset with a verifiable reward."""

    #: short identifier used in logs / run names (e.g. "gsm8k", "math")
    name: str = "base"

    def __init__(self, split: str = "train"):
        self.data = self._load(split)

    # ── Required per-benchmark implementations ──────────────────────────

    @abstractmethod
    def _load(self, split: str) -> list[dict]:
        """Return a list of item dicts with 'prompt' and 'answer' keys."""

    @abstractmethod
    def compute_reward(self, generated_text: str, ground_truth: str) -> float:
        """Binary (or bounded) verifiable reward for one generation."""

    # ── Shared behavior ──────────────────────────────────────────────────

    def get_prompt(self, item: dict) -> str:
        return item["prompt"]

    def get_ground_truth(self, item: dict) -> str:
        return item["answer"]

    def compute_group_rewards(
        self, generated_texts: list[str], ground_truth: str
    ) -> list[float]:
        """Reward each of the G rollouts for one prompt."""
        return [self.compute_reward(t, ground_truth) for t in generated_texts]

    # ── PyTorch Dataset protocol ─────────────────────────────────────────

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


In [ ]:
%%writefile data/gsm8k.py
"""
data/gsm8k.py

One job: load GSM8K and return formatted prompts ready for rollout generation.

Why GSM8K:
- Verifiable binary reward (correct numeric answer = 1.0, wrong = 0.0)
- No learned reward model needed — eliminates reward model variance
  as a confound in algorithm comparisons
- Short problems = manageable generation lengths on 6GB VRAM
- Standard benchmark — reviewers know it
"""

from datasets import load_dataset

from data.base_dataset import BaseRewardDataset
from rewards.gsm8k_reward import compute_reward


SYSTEM_PROMPT = (
    "Solve the following math problem step by step. "
    "At the end of your solution, write your final answer as: "
    "#### <number>"
)


def load_gsm8k(split: str = "train"):
    """
    Load GSM8K dataset.

    Args:
        split: "train" (7,473 problems) or "test" (1,319 problems)

    Returns:
        list of dicts, each with keys:
            "prompt":   formatted string ready for tokenization
            "answer":   ground truth numeric answer string (e.g. "42")
            "question": raw question text
    """
    raw = load_dataset("openai/gsm8k", "main", split=split)

    formatted = []
    for item in raw:
        prompt = f"{SYSTEM_PROMPT}\n\nQuestion: {item['question']}\n\nSolution:"
        # GSM8K answers are formatted as "...\n#### 42"
        # Extract just the numeric part after ####
        answer = item["answer"].split("####")[-1].strip()
        formatted.append({
            "prompt": prompt,
            "answer": answer,
            "question": item["question"],
        })

    return formatted


class GSM8KDataset(BaseRewardDataset):
    """
    GSM8K benchmark implementing the BaseRewardDataset interface.
    Reward: binary verifiable — extract final number, compare, 1.0/0.0.
    """

    name = "gsm8k"

    def _load(self, split: str) -> list[dict]:
        return load_gsm8k(split)

    def compute_reward(self, generated_text: str, ground_truth: str) -> float:
        return compute_reward(generated_text, ground_truth)


def get_prompt(item: dict) -> str:
    return item["prompt"]


def get_answer(item: dict) -> str:
    return item["answer"]


In [ ]:
%%writefile data/math_dataset.py
"""
data/math_dataset.py

One job: load the MATH competition benchmark (Hendrycks et al. 2021) and
return formatted prompts with verifiable boxed-answer ground truths.

Why MATH as the second benchmark:
- Harder than GSM8K (competition problems, 5 difficulty levels) —
  shows the algorithm ranking is not a GSM8K artifact
- Same verifiable-reward property: ground truth is the final \\boxed{}
  expression in the reference solution, so the reward stays binary and
  deterministic (no reward-model confound)

Source: EleutherAI/hendrycks_math — the maintained public mirror,
organized as one config per subject.
"""

from datasets import load_dataset

from data.base_dataset import BaseRewardDataset
from rewards.math_reward import compute_reward, extract_boxed


SYSTEM_PROMPT = (
    "Solve the following math problem step by step. "
    "At the end of your solution, write your final answer inside "
    "\\boxed{}."
)

MATH_SUBJECTS = [
    "algebra",
    "counting_and_probability",
    "geometry",
    "intermediate_algebra",
    "number_theory",
    "prealgebra",
    "precalculus",
]


def load_math(split: str = "train", subjects: list[str] | None = None):
    """
    Load the MATH benchmark.

    Args:
        split:    "train" (~7,500 problems) or "test" (~5,000 problems)
        subjects: optional subset of MATH_SUBJECTS (default: all)

    Returns:
        list of dicts with keys:
            "prompt":  formatted string ready for tokenization
            "answer":  ground-truth boxed expression (e.g. "\\frac{1}{2}")
            "level":   difficulty string (e.g. "Level 3")
            "subject": subject config name

    Problems whose reference solution has no parseable \\boxed{} answer
    are dropped — without a ground truth the binary reward is undefined.
    """
    formatted = []
    for subject in (subjects or MATH_SUBJECTS):
        raw = load_dataset("EleutherAI/hendrycks_math", subject, split=split)
        for item in raw:
            answer = extract_boxed(item["solution"])
            if answer is None:
                continue
            prompt = (f"{SYSTEM_PROMPT}\n\n"
                      f"Problem: {item['problem']}\n\nSolution:")
            formatted.append({
                "prompt": prompt,
                "answer": answer,
                "level": item.get("level", ""),
                "subject": subject,
            })

    return formatted


class MATHDataset(BaseRewardDataset):
    """
    MATH benchmark implementing the BaseRewardDataset interface.
    Reward: binary verifiable — extract last \\boxed{}, normalize, compare.
    """

    name = "math"

    def _load(self, split: str) -> list[dict]:
        return load_math(split)

    def compute_reward(self, generated_text: str, ground_truth: str) -> float:
        return compute_reward(generated_text, ground_truth)


In [ ]:
%%writefile losses/__init__.py
"""
losses/__init__.py

Explicit module imports — no wildcards.
Every module exports a compute() function; wildcard imports would collide.
The trainer imports these as modules: `from losses import ppo_loss`.
"""
from losses import ppo_loss, grpo_loss, dapo_loss, bapo_loss, sgrpo_loss

__all__ = ["ppo_loss", "grpo_loss", "dapo_loss", "bapo_loss", "sgrpo_loss"]


In [ ]:
%%writefile losses/bapo_loss.py
"""
losses/bapo_loss.py

BAPO: Balanced Policy Optimization with Adaptive Clipping.

One change from DAPO: the clipping bounds (c_low, c_high) are not fixed —
they are adjusted per batch so that positive-advantage tokens contribute
at least a target fraction rho_0 of the policy-gradient signal.

Why: fixed clipping treats positive and negative advantage updates
symmetrically. But negative-advantage tokens (suppressing bad behavior)
tend to dominate early in training, producing unstable, entropy-collapsing
gradients. BAPO rebalances this by widening c_high (letting more
positive-advantage tokens through) and, when that is exhausted, raising
c_low (clipping away more negative-advantage tokens).

Algorithm (verbatim from the BAPO paper):

    Input: movable range of clipping bounds [a-, b-] and [a+, b+],
           step size of upper bound d1, step size of lower bound d2,
           positive token contribution threshold rho_0

    Procedure Dynamically adjusting the clipping bounds c_high and c_low
        Initialize clipping bounds c_low = a- and c_high = a+
        while the positive token contribution rho < rho_0
              and c_low + d2 <= b- do
            if c_high + d1 <= b+ then
                c_high <- c_high + d1
            else
                c_low <- c_low + d2
        end

    Update the policy by maximizing:
        J_BAPO = E_y Sum_t min(r_t * A_t, clip(r_t, c_low, c_high) * A_t)

The positive token contribution rho is the share of the gradient-carrying
surrogate magnitude coming from positive-advantage tokens:

    rho = S+ / (S+ + S-)
    S+  = Sum over {A_t > 0, r_t < c_high} of  r_t * A_t        (unclipped)
    S-  = Sum over {A_t < 0, r_t > c_low}  of  r_t * |A_t|      (unclipped)

Tokens outside their bound are clipped: their surrogate is a constant and
carries no gradient, so they are excluded from rho. Widening c_high
un-clips more positive tokens (raises S+); raising c_low clips away more
negative tokens (lowers S-). Both moves push rho toward rho_0.

Note: BAPO is a standalone baseline in this project.
Its adaptive clipping is NOT composed with SGRPO's Future-KL weighting
because the adaptive bound derivation assumes uniform advantage weighting
per token — Future-KL changes this and the correct bounds under weighted
advantages have not been re-derived. Including them without justification
would be mathematically unjustified.

Returns:
    loss:       scalar loss
    advantages: [G] advantages for logging
    metrics:    dict of BAPO-specific metrics for WandB
    (or None if degenerate group)
"""

import torch
from losses.dapo_loss import is_degenerate_group
from losses.grpo_loss import compute_group_advantages, compute_kl_penalty_k3

# Safety cap on the bound-adjustment loop. With default ranges and step
# sizes the loop runs at most (b+ - a+)/d1 + (b- - a-)/d2 ~ 22 iterations;
# this cap only guards against misconfigured deltas (e.g. 0).
_MAX_BOUND_ITERS = 1000


def _positive_contribution(
    ratio: torch.Tensor,                 # [G, gen_len]
    advantages_expanded: torch.Tensor,   # [G, gen_len]
    attention_mask: torch.Tensor,        # [G, gen_len]
    c_low: float,
    c_high: float,
) -> float:
    """
    rho(c_low, c_high): share of gradient-carrying surrogate magnitude from
    positive-advantage tokens, given candidate clipping bounds.

    A token carries gradient only while its ratio is inside its clipping
    bound: positive-advantage tokens are clipped (gradient-dead) at
    r >= c_high, negative-advantage tokens at r <= c_low.
    """
    pos = (advantages_expanded > 0) & (ratio < c_high)
    neg = (advantages_expanded < 0) & (ratio > c_low)

    contrib = (ratio * advantages_expanded.abs()) * attention_mask
    s_pos = contrib[pos].sum().item() if pos.any() else 0.0
    s_neg = contrib[neg].sum().item() if neg.any() else 0.0

    return s_pos / (s_pos + s_neg + 1e-8)


def compute_adaptive_bounds(
    ratio: torch.Tensor,                 # [G, gen_len]
    advantages_expanded: torch.Tensor,   # [G, gen_len]
    attention_mask: torch.Tensor,        # [G, gen_len]
    c_low_min: float = 0.8,      # a-  (paper: start of lower-bound range)
    c_low_max: float = 0.9,      # b-  (paper: end of lower-bound range)
    c_high_min: float = 1.2,     # a+  (paper: start of upper-bound range)
    c_high_max: float = 1.32,    # b+  (paper: end of upper-bound range)
    delta_high: float = 0.01,    # d1  (paper: step size of upper bound)
    delta_low: float = 0.01,     # d2  (paper: step size of lower bound)
    rho_target: float = 0.5,     # rho_0 (paper: contribution threshold)
) -> tuple[float, float, float, int]:
    """
    The BAPO paper's "Dynamically adjusting the clipping bounds" procedure.

    Starts from the tightest bounds (a-, a+) and, while the positive token
    contribution rho is below rho_0, first widens c_high in steps of d1 up
    to b+, then raises c_low in steps of d2 up to b-. rho is recomputed
    after every adjustment because moving a bound changes which tokens are
    clipped.

    Returns:
        (c_low, c_high, rho, n_iterations)
    """
    c_low, c_high = c_low_min, c_high_min
    rho = _positive_contribution(
        ratio, advantages_expanded, attention_mask, c_low, c_high
    )

    iters = 0
    while (rho < rho_target
           and c_low + delta_low <= c_low_max
           and iters < _MAX_BOUND_ITERS):
        if c_high + delta_high <= c_high_max:
            c_high += delta_high
        else:
            c_low += delta_low
        rho = _positive_contribution(
            ratio, advantages_expanded, attention_mask, c_low, c_high
        )
        iters += 1

    return c_low, c_high, rho, iters


def compute(
    new_log_probs: torch.Tensor,    # [G, gen_len]
    old_log_probs: torch.Tensor,    # [G, gen_len]
    rewards: torch.Tensor,          # [G]
    attention_mask: torch.Tensor,   # [G, gen_len]
    c_low_min: float = 0.8,         # a-
    c_low_max: float = 0.9,         # b-
    c_high_min: float = 1.2,        # a+
    c_high_max: float = 1.32,       # b+
    delta_high: float = 0.01,       # d1
    delta_low: float = 0.01,        # d2
    rho_target: float = 0.5,        # rho_0
    ref_log_probs: torch.Tensor | None = None,  # [G, gen_len] — frozen reference
    kl_coef: float = 0.01,
) -> tuple[torch.Tensor, torch.Tensor, dict] | None:
    """
    BAPO loss: DAPO's token-normalized clipped surrogate with the paper's
    dynamically adjusted clipping bounds, plus the reference-model KL
    penalty (inherited from GRPO's foundation — the original BAPO paper
    keeps it).

    Note the bounds are ABSOLUTE ratio bounds clip(r, c_low, c_high),
    not the symmetric 1 +/- eps form — this is how the paper states the
    objective. Defaults (c_low in [0.8, 0.9], c_high in [1.2, 1.32])
    reduce to standard eps=0.2 clipping when no adjustment fires, with
    the upper range extending past DAPO's clip-higher value of 1.28.

    Returns None if group is degenerate.
    Returns (loss, advantages, metrics_dict) otherwise.
    """
    if is_degenerate_group(rewards):
        return None

    advantages = compute_group_advantages(rewards)
    ratio = torch.exp(new_log_probs - old_log_probs)
    advantages_expanded = advantages.unsqueeze(1).expand_as(ratio)

    # Bound search is a batch statistic, not part of the computation graph
    with torch.no_grad():
        c_low, c_high, rho, bound_iters = compute_adaptive_bounds(
            ratio, advantages_expanded, attention_mask,
            c_low_min=c_low_min, c_low_max=c_low_max,
            c_high_min=c_high_min, c_high_max=c_high_max,
            delta_high=delta_high, delta_low=delta_low,
            rho_target=rho_target,
        )

    unclipped = ratio * advantages_expanded
    clipped = torch.clamp(ratio, c_low, c_high) * advantages_expanded

    per_token_loss = -torch.min(unclipped, clipped)

    total_real_tokens = attention_mask.sum().clamp(min=1)
    policy_loss = (per_token_loss * attention_mask).sum() / total_real_tokens

    # KL penalty (k3), masked and token-normalized like the policy loss
    if ref_log_probs is not None:
        kl_per_token = compute_kl_penalty_k3(new_log_probs, ref_log_probs)
        kl_penalty = (kl_per_token * attention_mask).sum() / total_real_tokens
        loss = policy_loss + kl_coef * kl_penalty
    else:
        kl_penalty = torch.tensor(0.0, device=new_log_probs.device)
        loss = policy_loss

    # ── Metrics for WandB ────────────────────────────────────────────────
    with torch.no_grad():
        clip_fraction = (
            (ratio < c_low) | (ratio > c_high)
        ).float().mean().item()

        metrics = {
            "train/clip_fraction": clip_fraction,
            "train/ratio_mean": ratio.mean().item(),
            "train/ratio_std": ratio.std().item(),
            "train/ratio_max": ratio.max().item(),
            "train/ratio_min": ratio.min().item(),
            "train/approx_kl": (0.5 * (new_log_probs - old_log_probs).pow(2)).mean().item(),
            "bapo/c_low": c_low,
            "bapo/c_high": c_high,
            "bapo/bound_adjust_iterations": bound_iters,
            "bapo/positive_contribution_ratio": rho,
            "bapo/positive_contribution_target": rho_target,
            "bapo/contribution_gap": abs(rho - rho_target),
            "bapo/policy_loss": policy_loss.item(),
            "bapo/kl_penalty": kl_penalty.item(),
            "bapo/kl_coef": kl_coef,
            "bapo/has_ref_model": ref_log_probs is not None,
        }

    return loss, advantages, metrics


In [ ]:
%%writefile losses/dapo_loss.py
"""
losses/dapo_loss.py

DAPO: Direct Advantage Policy Optimization loss.

Two changes from GRPO:
1. Token-level normalization instead of sequence-level
   (longer responses get proportionally more gradient)
2. Dynamic sampling filter: skip groups where all rewards
   are identical (std=0 → degenerate advantage estimates)

No KL penalty — deliberately removed in the DAPO paper.

Returns:
    loss:       scalar loss
    advantages: [G] advantages for logging
    metrics:    dict of DAPO-specific metrics for WandB
    (or None if degenerate group)
"""

import torch
from losses.grpo_loss import compute_group_advantages


def is_degenerate_group(rewards: torch.Tensor) -> bool:
    """
    Returns True if all rewards in the group are identical.
    In this case, group-relative advantages are all zero — skip this group.
    This is DAPO's dynamic sampling filter.
    """
    return rewards.std().item() < 1e-8


def compute(
    new_log_probs: torch.Tensor,    # [G, gen_len]
    old_log_probs: torch.Tensor,    # [G, gen_len]
    rewards: torch.Tensor,          # [G]
    attention_mask: torch.Tensor,   # [G, gen_len] — 1 for real tokens, 0 for padding
    clip_epsilon: float = 0.2,
) -> tuple[torch.Tensor, torch.Tensor, dict] | None:
    """
    DAPO loss with token-level normalization and dynamic sampling filter.

    Returns None if group is degenerate (all same reward) — caller skips this batch.
    Returns (loss, advantages, metrics_dict) otherwise.

    Token-level normalization:
    Instead of mean() over [G, gen_len], divide by total number of real tokens.
    This gives longer correct responses more gradient weight — they contributed
    more tokens to the correct reasoning path.
    """
    if is_degenerate_group(rewards):
        return None

    advantages = compute_group_advantages(rewards)
    ratio = torch.exp(new_log_probs - old_log_probs)
    advantages_expanded = advantages.unsqueeze(1).expand_as(ratio)

    unclipped = ratio * advantages_expanded
    clipped = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon) * advantages_expanded

    per_token_loss = -torch.min(unclipped, clipped)

    # Token-level normalization: sum over real tokens, divide by total real token count
    total_real_tokens = attention_mask.sum().clamp(min=1)
    loss = (per_token_loss * attention_mask).sum() / total_real_tokens

    # ── Metrics for WandB ────────────────────────────────────────────────
    with torch.no_grad():
        clip_fraction = (
            (ratio < 1 - clip_epsilon) | (ratio > 1 + clip_epsilon)
        ).float().mean().item()

        metrics = {
            "train/clip_fraction": clip_fraction,
            "train/ratio_mean": ratio.mean().item(),
            "train/ratio_std": ratio.std().item(),
            "train/ratio_max": ratio.max().item(),
            "train/ratio_min": ratio.min().item(),
            "train/approx_kl": (0.5 * (new_log_probs - old_log_probs).pow(2)).mean().item(),
            "dapo/epsilon_low": clip_epsilon,
            "dapo/epsilon_high": clip_epsilon,  # symmetric in standard DAPO
            "dapo/token_level_loss": loss.item(),
            "dapo/kl_removed": True,
        }

    return loss, advantages, metrics


In [ ]:
%%writefile losses/grpo_loss.py
"""
losses/grpo_loss.py

GRPO: Group Relative Policy Optimization loss.
Adds group-relative advantage estimation on top of PPO's clipped surrogate.

Key difference from PPO:
- No value critic needed
- Advantages computed from reward differences within the group
- Baseline is the group mean reward, not a learned value function

KL penalty: the original GRPO (DeepSeekMath, Shao et al. 2024) includes
beta * KL(pi_theta || pi_ref) against a frozen reference model, using the
unbiased non-negative k3 estimator:

    D_KL = pi_ref/pi_theta - log(pi_ref/pi_theta) - 1
         = exp(ref_lp - new_lp) - (ref_lp - new_lp) - 1

Omitting it (as this code previously did) makes the baseline collapse on
long runs — which would invalidate any comparison against SGRPO.

Returns:
    loss:       scalar loss
    advantages: [G] advantages for logging
    metrics:    dict of GRPO-specific metrics for WandB
"""

import torch


def compute_group_advantages(rewards: torch.Tensor) -> torch.Tensor:
    """
    Compute group-relative advantages from a group of rewards.

    A_i = (r_i - mean(r)) / (std(r) + eps)

    Args:
        rewards: [G] tensor of scalar rewards, one per rollout

    Returns:
        advantages: [G] tensor, zero-mean, unit-variance within group

    Note: if std is zero (all rewards identical), advantages are all zero.
    This is the degenerate case DAPO's dynamic sampling filter prevents.
    """
    mean_r = rewards.mean()
    std_r = rewards.std() + 1e-8
    return (rewards - mean_r) / std_r


def compute_kl_penalty_k3(
    new_log_probs: torch.Tensor,    # [G, gen_len]
    ref_log_probs: torch.Tensor,    # [G, gen_len]
) -> torch.Tensor:
    """
    Per-token KL(pi_theta || pi_ref) via the k3 estimator (Schulman 2020):

        k3 = exp(ref_lp - new_lp) - (ref_lp - new_lp) - 1

    Unbiased and always >= 0. This is the exact form used in the GRPO
    paper (DeepSeekMath eq. 4). Shared by GRPO and BAPO.
    """
    log_ratio = (ref_log_probs - new_log_probs).clamp(-20.0, 20.0)
    return torch.exp(log_ratio) - log_ratio - 1.0


def compute(
    new_log_probs: torch.Tensor,    # [G, gen_len]
    old_log_probs: torch.Tensor,    # [G, gen_len]
    rewards: torch.Tensor,          # [G] — scalar reward per rollout
    clip_epsilon: float = 0.2,
    ref_log_probs: torch.Tensor | None = None,  # [G, gen_len] — frozen reference
    kl_coef: float = 0.01,
) -> tuple[torch.Tensor, torch.Tensor, dict]:
    """
    GRPO loss with group-relative advantage estimation and reference-model
    KL penalty (k3 estimator, per the DeepSeekMath paper).

    Returns:
        (loss, advantages, metrics_dict)
    """
    advantages = compute_group_advantages(rewards)

    ratio = torch.exp(new_log_probs - old_log_probs)
    advantages_expanded = advantages.unsqueeze(1).expand_as(ratio)

    unclipped = ratio * advantages_expanded
    clipped = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon) * advantages_expanded

    # Sequence-level normalization (divide by G, average over tokens per sequence)
    policy_loss = -torch.min(unclipped, clipped).mean()

    if ref_log_probs is not None:
        kl_penalty = compute_kl_penalty_k3(new_log_probs, ref_log_probs).mean()
        loss = policy_loss + kl_coef * kl_penalty
    else:
        kl_penalty = torch.tensor(0.0, device=new_log_probs.device)
        loss = policy_loss

    # ── Metrics for WandB ────────────────────────────────────────────────
    with torch.no_grad():
        G = rewards.shape[0]
        clip_fraction = (
            (ratio < 1 - clip_epsilon) | (ratio > 1 + clip_epsilon)
        ).float().mean().item()

        metrics = {
            "train/clip_fraction": clip_fraction,
            "train/ratio_mean": ratio.mean().item(),
            "train/ratio_std": ratio.std().item(),
            "train/ratio_max": ratio.max().item(),
            "train/ratio_min": ratio.min().item(),
            "train/approx_kl": (0.5 * (new_log_probs - old_log_probs).pow(2)).mean().item(),
            "grpo/group_size": G,
            "grpo/group_reward_mean": rewards.mean().item(),
            "grpo/group_reward_std": rewards.std().item(),
            "grpo/group_reward_range": (rewards.max() - rewards.min()).item(),
            "grpo/num_groups": 1,  # per-step, always 1 group
            "grpo/policy_loss": policy_loss.item(),
            "grpo/kl_penalty": kl_penalty.item(),
            "grpo/kl_coef": kl_coef,
            "grpo/has_ref_model": ref_log_probs is not None,
        }

    return loss, advantages, metrics


In [ ]:
%%writefile losses/ppo_loss.py
"""
losses/ppo_loss.py

RLHF PPO clipped surrogate loss.

Role in this project: infrastructure validator.
If this trains and reward goes up, the shared trainer is confirmed working.
Every bug found here is a bug not found later inside SGRPO's math.

KL penalty: since this project is RL-for-LLMs, we use RLHF PPO
(InstructGPT / Ouyang et al. 2022), which adds beta * KL(pi_theta || pi_ref)
against a frozen copy of the pretrained model to prevent the policy from
drifting away from the pretrained distribution. Vanilla PPO (Schulman 2017)
has no such penalty; using it as an LLM baseline would let it collapse on
long runs and invalidate the comparison.

Returns:
    loss:       scalar loss
    advantages: [G] advantages for logging
    metrics:    dict of PPO-specific metrics for WandB
"""

import torch


def compute(
    new_log_probs: torch.Tensor,    # [G, gen_len] — log probs under current policy
    old_log_probs: torch.Tensor,    # [G, gen_len] — log probs stored during rollout
    advantages: torch.Tensor,       # [G] — one advantage per rollout
    clip_epsilon: float = 0.2,
    ref_log_probs: torch.Tensor | None = None,  # [G, gen_len] — frozen reference model
    kl_coef: float = 0.01,
) -> tuple[torch.Tensor, torch.Tensor, dict]:
    """
    RLHF PPO clipped surrogate objective.

    r_t = pi_theta(a_t) / pi_theta_old(a_t) = exp(new_log_prob - old_log_prob)
    L = -E[min(r_t * A, clip(r_t, 1-eps, 1+eps) * A)] + beta * KL(pi_theta || pi_ref)

    KL uses the k1 estimator E[log pi_theta - log pi_ref] as in InstructGPT.

    Returns:
        (loss, advantages, metrics_dict)
    """
    # Importance sampling ratio: [G, gen_len]
    ratio = torch.exp(new_log_probs - old_log_probs)

    # Expand advantages from [G] to [G, gen_len] for token-level multiplication
    advantages_expanded = advantages.unsqueeze(1).expand_as(ratio)

    # Clipped surrogate
    unclipped = ratio * advantages_expanded
    clipped = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon) * advantages_expanded

    policy_loss = -torch.min(unclipped, clipped).mean()

    # RLHF KL penalty against frozen reference model (k1 estimator)
    if ref_log_probs is not None:
        kl_penalty = (new_log_probs - ref_log_probs).mean()
        loss = policy_loss + kl_coef * kl_penalty
    else:
        kl_penalty = torch.tensor(0.0, device=new_log_probs.device)
        loss = policy_loss

    # ── Metrics for WandB ────────────────────────────────────────────────
    with torch.no_grad():
        clip_fraction = (
            (ratio < 1 - clip_epsilon) | (ratio > 1 + clip_epsilon)
        ).float().mean().item()

        metrics = {
            "train/clip_fraction": clip_fraction,
            "train/ratio_mean": ratio.mean().item(),
            "train/ratio_std": ratio.std().item(),
            "train/ratio_max": ratio.max().item(),
            "train/ratio_min": ratio.min().item(),
            "train/approx_kl": (0.5 * (new_log_probs - old_log_probs).pow(2)).mean().item(),
            "ppo/policy_loss": policy_loss.item(),
            "ppo/kl_penalty": kl_penalty.item(),
            "ppo/kl_coef": kl_coef,
            "ppo/has_ref_model": ref_log_probs is not None,
        }

    return loss, advantages, metrics


In [ ]:
%%writefile losses/sgrpo_loss.py
"""
losses/sgrpo_loss.py

SGRPO: State-aware Group Relative Policy Optimization loss.

Components:
1. Group-relative advantage estimation (from GRPO)
   — computed from contamination-free rollouts (via sgrpo_rollout.py)
2. Token-level normalization (from DAPO)
3. Dynamic sampling filter (from DAPO)
4. Future-KL token weighting (from FIPO, cited)
   — operates at optimization phase, orthogonal to state isolation

The complete SGRPO objective:

    L^SGRPO = -(1 / Σ|o_i|) Σ_i Σ_t min(r_t · Â_t^ISO, clip(r_t, 1-ε, 1+ε) · Â_t^ISO)

where:
    r_t = π_θ(o_{i,t} | q, o_{i,<t}) / π_{θ_old}(o_{i,t} | q, o_{i,<t})
    Â_t^ISO = A_i^ISO · w(FutureKL_t)
    A_i^ISO = (r_i - mean(r_j)) / std(r_j)

The superscript ISO denotes that advantages were estimated under the i.i.d.
guarantee restored by state isolation in rollouts/sgrpo_rollout.py.

What is NOT included and why:
- BAPO's adaptive clipping: mathematically incompatible with Future-KL weighting
  without re-deriving optimal bounds under weighted advantages (future work)
- Frozen reference model KL penalty: not needed — Future-KL uses stored
  rollout log_probs (pi_theta_old), not a separate frozen model

Returns:
    loss:              scalar loss
    advantages:        [G] advantages for logging
    metrics:           dict with SGRPO-specific metrics for WandB
    (or None if degenerate group)
"""

import torch
from losses.dapo_loss import is_degenerate_group
from losses.grpo_loss import compute_group_advantages


def _reverse_discounted_cumsum(
    delta: torch.Tensor,    # [G, gen_len]
    gamma: float,
    chunk_size: int = 512,
) -> torch.Tensor:
    """
    out[t] = Σ_{k=t}^{T-1} γ^{k-t} · delta[k]  — a right-to-left discounted
    scan, computed without a per-token Python loop.

    Within a chunk of length L the scan collapses to one cumsum:
        out[t] = (Σ_{k=t}^{e-1} γ^k · delta[k]) / γ^t  +  γ^{e-t} · carry
    Only the carry between chunks is sequential, so the Python loop runs
    gen_len / chunk_size times (4 iterations for a 2048-token generation)
    instead of gen_len times.

    Chunking is also what makes the γ-power trick numerically safe: the
    powers γ^j inside one chunk span at most chunk_size exponents
    (2^(512/30) ≈ 2^17 at decay=30), far from fp32 overflow — whereas a
    single global cumsum over thousands of tokens would need γ^{-t} up to
    2^(t/30) and overflow.
    """
    G, T = delta.shape
    orig_dtype = delta.dtype
    delta = delta.float()

    out = torch.empty_like(delta)
    carry = torch.zeros(G, 1, dtype=delta.dtype, device=delta.device)

    first_start = ((T - 1) // chunk_size) * chunk_size
    for start in range(first_start, -1, -chunk_size):
        end = min(start + chunk_size, T)
        L = end - start
        j = torch.arange(L, device=delta.device, dtype=delta.dtype)
        gpow = gamma ** j                                       # [L]
        weighted = delta[:, start:end] * gpow                   # [G, L]
        # c[j] = Σ_{m=j}^{L-1} γ^m · delta[start+m]
        c = torch.flip(torch.cumsum(torch.flip(weighted, dims=[1]), dim=1),
                       dims=[1])
        out[:, start:end] = c / gpow + carry * gamma ** (L - j)
        carry = out[:, start:start + 1]

    return out.to(orig_dtype)


def compute_future_kl(
    new_log_probs: torch.Tensor,    # [G, gen_len]
    old_log_probs: torch.Tensor,    # [G, gen_len]
    attention_mask: torch.Tensor,   # [G, gen_len]
    decay_rate: float = 30.0,
) -> torch.Tensor:
    """
    Compute Future-KL influence weights per token (vectorized).

    From FIPO (Ma et al. 2026):
    FutureKL_t = Σ_{k=t}^{T} γ^{k-t} · M_k · δlog_p_k

    where:
    - δlog_p_k = log π_θ(y_k) - log π_{θ_old}(y_k)
    - γ = 2^{-1/decay_rate}
    - M_k = attention_mask (1 for real tokens, 0 for padding)

    Padding is zeroed BEFORE the scan (the "tensor trap"): a padded token
    must contribute nothing to any earlier token's sum, and masking after
    the scan would not undo its contribution to the running discount.

    Implementation: chunked reverse discounted cumsum — see
    _reverse_discounted_cumsum. O(gen_len / chunk_size) sequential steps,
    everything else parallel on GPU.

    Args:
        new_log_probs:  [G, gen_len] — current policy
        old_log_probs:  [G, gen_len] — stored rollout policy
        attention_mask: [G, gen_len]
        decay_rate:     controls how far future tokens influence current token
                        larger = more future context, smaller = more local

    Returns:
        future_kl: [G, gen_len] — influence weight per token
    """
    gamma = 2.0 ** (-1.0 / decay_rate)

    # Clamp log-prob differences to prevent exp() overflow
    # |δ| > 20 means ratio > 5e8 — numerically meaningless
    delta_log_p = (new_log_probs - old_log_probs).clamp(-20.0, 20.0)
    delta_log_p = delta_log_p * attention_mask

    return _reverse_discounted_cumsum(delta_log_p, gamma)


def compute_influence_weights(
    future_kl: torch.Tensor,       # [G, gen_len]
    clip_low: float = 1.0,
    clip_high: float = 1.2,
) -> torch.Tensor:
    """
    Convert Future-KL signal to influence weights via clipping.

    From FIPO: weights are clipped to [clip_low, clip_high] to prevent
    extreme weighting of any single token from destabilizing training.

    w_t = clip(exp(future_kl_t), clip_low, clip_high)

    Numerical stability: clamp future_kl before exp() to prevent overflow.
    exp(20) ≈ 5e8 which is already well beyond clip_high.

    Args:
        future_kl:  [G, gen_len]
        clip_low:   minimum weight (default 1.0 from FIPO paper)
        clip_high:  maximum weight (default 1.2 from FIPO paper)

    Returns:
        weights: [G, gen_len], each in [clip_low, clip_high]
    """
    # Clamp input to prevent exp() overflow — any value > log(clip_high)
    # would be clipped anyway, so no information loss
    safe_kl = future_kl.clamp(-10.0, 10.0)
    weights = torch.exp(safe_kl)
    weights = torch.clamp(weights, clip_low, clip_high)
    return weights


def compute(
    new_log_probs: torch.Tensor,    # [G, gen_len]
    old_log_probs: torch.Tensor,    # [G, gen_len]
    rewards: torch.Tensor,          # [G]
    attention_mask: torch.Tensor,   # [G, gen_len]
    clip_epsilon: float = 0.2,
    future_kl_decay: float = 30.0,
    future_kl_clip_low: float = 1.0,
    future_kl_clip_high: float = 1.2,
) -> tuple[torch.Tensor, torch.Tensor, dict] | None:
    """
    SGRPO loss.

    The complete objective:
    L = -(1 / Σ|o_i|) · Σ_{i,t} [
        min(r_t · A_i^ISO · w_t, clip(r_t, 1-ε, 1+ε) · A_i^ISO · w_t)
    ]

    where:
    - A_i^ISO = group-relative advantage from contamination-free rollouts
    - w_t = Future-KL influence weight at token t
    - r_t = importance sampling ratio at token t

    Returns None if group is degenerate.
    Returns (loss, advantages, metrics_dict) otherwise.
    """
    if is_degenerate_group(rewards):
        return None

    # Group-relative advantages from isolated rollouts
    # (the "ISO" superscript — isolation happened in the rollout generator)
    advantages = compute_group_advantages(rewards)

    # Importance sampling ratio with numerical stability.
    # This DOES require grad: d r_t / d theta is the intended policy gradient.
    log_ratio = (new_log_probs - old_log_probs).clamp(-20.0, 20.0)
    ratio = torch.exp(log_ratio)

    # Future-KL influence weights.
    # w_t scales the advantage, so it is an advantage-like coefficient and
    # MUST be treated as a constant w.r.t. theta — exactly as `advantages`
    # itself is. Computing it under no_grad both (a) prevents an unintended
    # second gradient term  r_t · A_i · ∇_θ w_t  from leaking into the policy
    # gradient wherever the clip is not saturating, and (b) avoids building
    # the autograd graph for the reverse-cumsum scan at all (nothing to free
    # later). `ratio` above is built from `new_log_probs` OUTSIDE this block,
    # so the intended gradient  ∇_θ r_t · A_i · w_t  is fully preserved.
    with torch.no_grad():
        future_kl = compute_future_kl(
            new_log_probs, old_log_probs, attention_mask, future_kl_decay
        )
        influence_weights = compute_influence_weights(
            future_kl, future_kl_clip_low, future_kl_clip_high
        )

    # Weighted advantages: A_i^ISO * w_t
    advantages_expanded = advantages.unsqueeze(1).expand_as(ratio)
    weighted_advantages = advantages_expanded * influence_weights

    # Clipped surrogate with weighted advantages
    unclipped = ratio * weighted_advantages
    clipped = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon) * weighted_advantages

    per_token_loss = -torch.min(unclipped, clipped)

    # Token-level normalization (from DAPO)
    total_real_tokens = attention_mask.sum().clamp(min=1)
    loss = (per_token_loss * attention_mask).sum() / total_real_tokens

    # ── Metrics for WandB ────────────────────────────────────────────────
    with torch.no_grad():
        clip_fraction = (
            (ratio < 1 - clip_epsilon) | (ratio > 1 + clip_epsilon)
        ).float().mean().item()

        pos_adv = (advantages > 0).float().sum().item()
        total_adv = advantages.numel()

        metrics = {
            # Universal
            "train/clip_fraction": clip_fraction,
            "train/ratio_mean": ratio.mean().item(),
            "train/ratio_std": ratio.std().item(),
            "train/ratio_max": ratio.max().item(),
            "train/ratio_min": ratio.min().item(),
            "train/approx_kl": (0.5 * log_ratio.pow(2)).mean().item(),
            "train/positive_advantage_ratio": pos_adv / max(total_adv, 1),
            # SGRPO-specific
            "sgrpo/future_kl_mean": future_kl.mean().item(),
            "sgrpo/future_kl_std": future_kl.std().item(),
            "sgrpo/future_kl_max": future_kl.max().item(),
            "sgrpo/future_kl_min": future_kl.min().item(),
            "sgrpo/influence_weight_mean": influence_weights.mean().item(),
            "sgrpo/influence_weight_std": influence_weights.std().item(),
            "sgrpo/weighted_advantage_mean": weighted_advantages.mean().item(),
            "sgrpo/weighted_advantage_std": weighted_advantages.std().item(),
            "sgrpo/decay_rate": future_kl_decay,
            "sgrpo/clip_low": future_kl_clip_low,
            "sgrpo/clip_high": future_kl_clip_high,
        }

    return loss, advantages, metrics


In [ ]:
%%writefile models/__init__.py


In [ ]:
%%writefile models/load_model.py
"""
models/load_model.py

One job: load any HuggingFace causal LM and return (model, tokenizer).

Universal loader: supports pure-SSM models (Mamba-1/2), hybrid SSM+attention
models (Jamba, Zamba), and standard Transformers (Qwen, Phi, Gemma, ...).
The architecture class determines whether SGRPO's state isolation applies:

- "ssm":         all layers carry recurrent state → isolation required
- "hybrid":      SSM layers carry state, attention layers are stateless
                 → isolation of SSM layers required (future work)
- "transformer": stateless between generate() calls → isolation is a no-op;
                 SGRPO degenerates to GRPO (this is the paper's control)

Default remains state-spaces/mamba-130m-hf:
- Fits in 6GB VRAM during training at batch_size=1
- Fast enough for rapid iteration on loss function math
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "state-spaces/mamba-130m-hf"

# Architecture detection by model class name
SSM_ARCHITECTURES = {
    "MambaForCausalLM",
    "Mamba2ForCausalLM",
    "FalconMambaForCausalLM",
}
HYBRID_ARCHITECTURES = {
    "JambaForCausalLM",
    "ZambaForCausalLM",
    "Zamba2ForCausalLM",
    "BambaForCausalLM",
    "NemotronHForCausalLM",
}


def detect_architecture(model) -> str:
    """
    Return 'ssm', 'hybrid', or 'transformer' for a loaded model.
    This determines whether SGRPO's state isolation is needed.
    """
    class_name = model.__class__.__name__
    if class_name in SSM_ARCHITECTURES:
        return "ssm"
    if class_name in HYBRID_ARCHITECTURES:
        return "hybrid"
    return "transformer"


def load_model(device: str = "cuda", dtype: str = "bfloat16",
               model_name: str = MODEL_NAME):
    """
    Load a causal LM and tokenizer.

    Args:
        device:     "cuda" or "cpu"
        dtype:      "bfloat16" (default, saves VRAM) or "float32" (debugging)
        model_name: any HuggingFace model name or local path.
                    Default: state-spaces/mamba-130m-hf

    Returns:
        model:     causal LM moved to device, eval mode by default
        tokenizer: AutoTokenizer with pad_token set

    Use detect_architecture(model) to decide whether state isolation applies.

    VRAM cost for mamba-130m at bfloat16: ~260MB for weights alone.
    Training overhead (optimizer + gradients + activations): ~3-4GB total.
    Fits within RTX 4060 6GB at batch_size=1, grad_accum=8.
    """
    torch_dtype = torch.bfloat16 if dtype == "bfloat16" else torch.float32

    print(f"Loading {model_name} on {device} in {dtype}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Many tokenizers (Mamba's included) ship without a pad_token — this
    # causes silent errors in batched generation. Set it explicitly.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch_dtype,
    ).to(device)

    arch = detect_architecture(model)
    print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,} "
          f"| Architecture: {arch}")
    if torch.cuda.is_available() and device != "cpu":
        print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

    return model, tokenizer


def get_model_name() -> str:
    return MODEL_NAME


In [ ]:
%%writefile rewards/__init__.py


In [ ]:
%%writefile rewards/gsm8k_reward.py
"""
rewards/gsm8k_reward.py

One job: given a generated response and a ground truth answer,
return 1.0 if correct, 0.0 if not.

This is the ONLY reward function in this project.
No learned reward model. No ambiguity. Binary and verifiable.

Why this matters for research validity:
If reward is ambiguous, differences in algorithm performance
could come from reward variance, not algorithm quality.
Binary verifiable reward eliminates this confound entirely.
"""

import re


def extract_answer(text: str) -> str | None:
    """
    Extract the numeric answer from a generated response.

    Looks for patterns in order of specificity:
    1. #### <number>  (GSM8K standard format)
    2. The answer is <number>
    3. = <number> at end of text
    4. Last standalone number in the text (fallback)

    Returns the extracted number as a string, or None if nothing found.
    """
    # Pattern 1: GSM8K standard #### format
    match = re.search(r"####\s*([\d,]+\.?\d*)", text)
    if match:
        return match.group(1).replace(",", "").strip()

    # Pattern 2: "the answer is X"
    match = re.search(r"[Tt]he answer is\s*([\d,]+\.?\d*)", text)
    if match:
        return match.group(1).replace(",", "").strip()

    # Pattern 3: "= X" near end of text
    match = re.search(r"=\s*([\d,]+\.?\d*)\s*$", text.strip())
    if match:
        return match.group(1).replace(",", "").strip()

    # Pattern 4: last standalone number (fallback — less reliable)
    numbers = re.findall(r"\b(\d+(?:,\d{3})*(?:\.\d+)?)\b", text)
    if numbers:
        return numbers[-1].replace(",", "").strip()

    return None


def compute_reward(generated_text: str, ground_truth_answer: str) -> float:
    """
    Compute binary reward for a generated response.

    Args:
        generated_text:      full text generated by the model
        ground_truth_answer: correct answer string from GSM8K (e.g. "42")

    Returns:
        1.0 if the extracted answer matches ground truth, else 0.0

    Note: comparison is done after stripping whitespace and
    normalizing decimal representation (42 == 42.0 == 42.00).
    """
    extracted = extract_answer(generated_text)

    if extracted is None:
        return 0.0

    # Normalize both to float for comparison
    # handles cases like "42" vs "42.0" vs "42.00"
    try:
        extracted_float = float(extracted)
        truth_float = float(ground_truth_answer.replace(",", "").strip())
        return 1.0 if abs(extracted_float - truth_float) < 1e-6 else 0.0
    except ValueError:
        # If either can't be converted to float, do string comparison
        return 1.0 if extracted.strip() == ground_truth_answer.strip() else 0.0


def compute_group_rewards(
    generated_texts: list[str],
    ground_truth_answer: str
) -> list[float]:
    """
    Compute rewards for a group of G rollouts for the same prompt.
    Used by the group-relative advantage estimator.

    Args:
        generated_texts:     list of G generated responses
        ground_truth_answer: single correct answer (same for all rollouts)

    Returns:
        list of G floats, each 0.0 or 1.0
    """
    return [compute_reward(text, ground_truth_answer) for text in generated_texts]


In [ ]:
%%writefile rewards/math_reward.py
"""
rewards/math_reward.py

One job: binary verifiable reward for the MATH (Hendrycks et al. 2021)
competition benchmark.

Ground truth on MATH is the content of the final \\boxed{...} in the
reference solution. The model is instructed to end its solution with
\\boxed{<answer>}; we extract the last boxed expression from the
generation, normalize both sides, and compare exactly.

Same design contract as gsm8k_reward: deterministic, no learned reward
model, 1.0/0.0 only — reward variance must not confound the algorithm
comparison.
"""

import re


def extract_boxed(text: str) -> str | None:
    """
    Return the content of the LAST \\boxed{...} in the text, handling
    nested braces (e.g. \\boxed{\\frac{1}{2}}), or None if absent.
    """
    marker = r"\boxed{"
    start = text.rfind(marker)
    if start == -1:
        return None

    i = start + len(marker)
    depth = 1
    out = []
    while i < len(text) and depth > 0:
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                break
        out.append(ch)
        i += 1

    return "".join(out) if depth == 0 else None


def normalize_answer(answer: str) -> str:
    """
    Canonicalize a MATH answer string for exact comparison.

    Covers the common notational variance between model output and the
    reference solutions: spacing commands, \\left/\\right, \\dfrac vs
    \\frac, \\text{} wrappers, surrounding $ signs, trailing units-like
    text, and thousands separators in plain numbers. This is the standard
    lightweight normalizer used by MATH evaluation harnesses — not a CAS;
    symbolically equal but textually different answers score 0, identically
    for every algorithm, so the comparison stays fair.
    """
    s = answer.strip()

    # Strip surrounding $ ... $ and trailing period
    s = s.strip("$").strip()
    s = s.rstrip(".")

    # LaTeX cosmetic commands that don't change the value
    s = s.replace(r"\left", "").replace(r"\right", "")
    s = s.replace(r"\!", "").replace(r"\,", "").replace(r"\;", "")
    s = s.replace(r"\dfrac", r"\frac").replace(r"\tfrac", r"\frac")

    # \text{...} wrappers (units, words)
    s = re.sub(r"\\text\{([^{}]*)\}", r"\1", s)

    # \frac{a}{b} with single-char args sometimes appears as \frac ab
    s = re.sub(r"\\frac\s*([0-9a-z])\s*([0-9a-z])", r"\\frac{\1}{\2}", s)

    # Remove all whitespace
    s = re.sub(r"\s+", "", s)

    # Thousands separators in plain numbers: 1,000 -> 1000
    if re.fullmatch(r"-?[\d,]+(\.\d+)?", s):
        s = s.replace(",", "")

    return s


def compute_reward(generated_text: str, ground_truth_answer: str) -> float:
    """
    1.0 if the last \\boxed{} in the generation matches the ground truth
    after normalization, else 0.0. Numeric answers also match across
    representation (7 == 7.0).
    """
    extracted = extract_boxed(generated_text)
    if extracted is None:
        return 0.0

    a = normalize_answer(extracted)
    b = normalize_answer(ground_truth_answer)

    if a == b:
        return 1.0

    # Numeric fallback: 7 vs 7.0 vs 7.00
    try:
        return 1.0 if abs(float(a) - float(b)) < 1e-6 else 0.0
    except ValueError:
        return 0.0


def compute_group_rewards(
    generated_texts: list[str],
    ground_truth_answer: str,
) -> list[float]:
    """Reward each of the G rollouts for one prompt."""
    return [compute_reward(t, ground_truth_answer) for t in generated_texts]


In [ ]:
%%writefile rollouts/__init__.py


In [ ]:
%%writefile rollouts/base_rollout.py
"""
rollouts/base_rollout.py

One job: generate G rollouts for a given prompt.
Standard rollout — no SSM state isolation.
Used by PPO, GRPO, DAPO, BAPO.

Returns tokens and log_probs needed for the loss function.
The log_probs returned here are the OLD log_probs (pi_theta_old)
used in the importance sampling ratio during optimization.
"""

import torch
from transformers import MambaForCausalLM, AutoTokenizer


def generate_rollouts(
    model: MambaForCausalLM,
    tokenizer: AutoTokenizer,
    prompt: str,
    group_size: int = 4,
    max_new_tokens: int = 256,
    temperature: float = 1.0,
    device: str = "cuda",
) -> dict:
    """
    Generate group_size rollouts for a single prompt.

    Args:
        model:          the current policy model (pi_theta_old during rollout)
        tokenizer:      tokenizer
        prompt:         formatted prompt string
        group_size:     G — number of rollouts to generate
        max_new_tokens: maximum tokens to generate per rollout
        temperature:    sampling temperature (1.0 = standard sampling)
        device:         "cuda" or "cpu"

    Returns dict with:
        "input_ids":        [G, prompt_len] — prompt token IDs
        "generated_ids":    [G, prompt_len + gen_len] — full sequence IDs
        "generated_texts":  list of G decoded strings (generated portion only)
        "old_log_probs":    [G, gen_len] — log probs under pi_theta_old
                            CRITICAL: stored BEFORE any gradient update.
                            Used in importance sampling ratio during optimization.
    """
    model.eval()

    # Tokenize prompt
    prompt_inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(device)

    prompt_len = prompt_inputs.input_ids.shape[1]

    all_generated_ids = []
    all_old_log_probs = []
    all_generated_texts = []

    with torch.no_grad():
        for k in range(group_size):
            # Generate one rollout
            # NOTE: for Mamba, generate() re-encodes prompt each time
            # because no cache is passed — this means no state contamination
            # in base_rollout (each generate() call starts fresh).
            # The contamination issue happens when you manually manage
            # the cache across rollouts — which naive GRPO implementations do.
            output = model.generate(
                prompt_inputs.input_ids,
                attention_mask=prompt_inputs.attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                pad_token_id=tokenizer.pad_token_id,
                return_dict_in_generate=True,
                output_scores=True,
            )

            generated_ids = output.sequences  # [1, prompt_len + gen_len]
            scores = output.scores            # tuple of [1, vocab_size] per generated token

            # Compute log_probs for each generated token
            # scores[t] is the logit distribution at generation step t
            gen_len = len(scores)
            log_probs = []
            for t in range(gen_len):
                token_id = generated_ids[0, prompt_len + t]
                log_prob = torch.log_softmax(scores[t][0], dim=-1)[token_id]
                log_probs.append(log_prob)

            old_log_probs = torch.stack(log_probs)  # [gen_len]

            # Decode only the generated portion
            generated_text = tokenizer.decode(
                generated_ids[0, prompt_len:],
                skip_special_tokens=True
            )

            all_generated_ids.append(generated_ids[0])
            all_old_log_probs.append(old_log_probs)
            all_generated_texts.append(generated_text)

    # Rollouts stop at EOS independently, so lengths differ within the
    # group — pad to the longest before stacking. Padded log-prob positions
    # are zeros; the trainer's attention mask excludes them from the loss.
    padded_log_probs = []
    max_lp_len = max(lp.shape[0] for lp in all_old_log_probs)
    for lp in all_old_log_probs:
        pad_len = max_lp_len - lp.shape[0]
        if pad_len > 0:
            lp = torch.cat([lp, torch.zeros(pad_len, device=lp.device,
                                            dtype=lp.dtype)])
        padded_log_probs.append(lp)

    return {
        "input_ids": prompt_inputs.input_ids.expand(group_size, -1),
        "generated_ids": torch.nn.utils.rnn.pad_sequence(
            all_generated_ids, batch_first=True,
            padding_value=tokenizer.pad_token_id,
        ),                                                       # [G, full_len]
        "generated_texts": all_generated_texts,                  # list of G strings
        "old_log_probs": torch.stack(padded_log_probs),          # [G, gen_len]
        "prompt_len": prompt_len,
    }


In [ ]:
%%writefile rollouts/sgrpo_rollout.py
"""
rollouts/sgrpo_rollout.py

One job: generate G rollouts with SSM state isolation between each rollout.
This is SGRPO's core novel contribution.

The mechanism:
1. Process the prompt through the model, capture SSM hidden state h_0
2. Before each rollout, restore h_0 exactly
3. Generate rollout from clean starting state
4. Repeat for all G rollouts

Why this restores the i.i.d. assumption:
- Without isolation: rollout k starts from h_T of rollout k-1 (contaminated)
- With isolation:    rollout k starts from h_0 (clean, identical for all k)
- The i.i.d. assumption requires identical starting conditions per rollout

Cache API (transformers >= 4.46, verified on 5.13):
- `MambaCache` was removed. The model returns a universal
  `transformers.cache_utils.DynamicCache` in `output.cache_params`.
- Per-layer state lives in `cache.layers[i]`:
    - recurrent_states: [batch, d_inner, d_state]  — the SSM state h_t
    - conv_states:      [batch, d_inner, conv_kernel] — causal-conv buffer
- `cache_position` no longer exists in the Mamba forward signature; the
  model switches prefill/decode via `cache.has_previous_state(layer_idx)`.

Complete isolation requires restoring BOTH tensors: recurrent_states carries
the SSM recurrence h_t, and conv_states carries the short causal-convolution
context (the last `conv_kernel` token projections). Restoring only the SSM
state would leave the first few generated tokens conditioned on the previous
rollout's conv buffer.

Hardening features:
- SSM state isolation diagnostics (cosine similarity verification)
- Single cache reused across rollouts — restore is a pure tensor copy,
  no extra prompt forward passes (keeps the overhead claim honest)
- Timing instrumentation for overhead measurement
- Numerical stability in log-probability computation
"""

import time

import torch
from transformers import MambaForCausalLM, AutoTokenizer
from transformers.cache_utils import DynamicCache


def _snapshot_ssm_states(cache: DynamicCache) -> list[dict]:
    """
    Extract and clone the SSM recurrent states AND conv states of every
    layer that carries them.

    Per layer for Mamba-130m (24 layers):
        recurrent_states: [1, 1536, 16] — h_t, the tensor that carries
                          contamination between rollouts
        conv_states:      [1, 1536, 4]  — causal conv buffer

    Hybrid SSM+attention models (Jamba, Zamba, Bamba, Nemotron-H) mix SSM
    layers with attention layers in the same cache; attention layers have
    no recurrent state and are skipped here (their KV entries are handled
    separately in _restore_ssm_states). Each snapshot entry records its
    layer index so restore targets the right layers.

    NOT the residual stream hidden states (those are output_hidden_states,
    which misleadingly show cosine similarity ~1.0 — see CLAUDE.md §2.4).
    """
    snapshots = []
    for layer_idx, layer in enumerate(cache.layers):
        recurrent = getattr(layer, "recurrent_states", None)
        if recurrent is None:
            continue  # attention layer in a hybrid model
        conv = getattr(layer, "conv_states", None)
        snapshots.append({
            "layer_idx": layer_idx,
            "recurrent": recurrent.detach().clone(),
            "conv": conv.detach().clone() if conv is not None else None,
        })

    if not snapshots:
        raise RuntimeError(
            "No SSM recurrent states found in the cache — this model has no "
            "state to isolate (pure transformer?). Use "
            "base_rollout.generate_rollouts instead; the trainer's "
            "architecture detection should already route it there."
        )
    return snapshots


def _restore_ssm_states(
    cache: DynamicCache,
    snapshot: list[dict],
    prompt_len: int | None = None,
) -> None:
    """
    Restore SSM states from a snapshot into a DynamicCache in-place.
    This is the actual state isolation operation.

    Cost: L * d_inner * (d_state + conv_kernel) floats copied
    For Mamba-130m: 24 * 1536 * (16 + 4) = 737,280 floats ≈ 1.4MB in fp32.
    Negligible overhead; no forward passes involved.

    Hybrid models: SSM layers are restored by tensor copy; the remaining
    attention layers still hold KV entries appended by the previous rollout,
    so the cache is cropped back to the prompt length. (Attention layers are
    stateless in the sense that cropped KV is bit-identical to re-encoding —
    no approximation involved.)
    """
    for state in snapshot:
        layer = cache.layers[state["layer_idx"]]
        layer.recurrent_states.copy_(state["recurrent"])
        if state["conv"] is not None:
            layer.conv_states.copy_(state["conv"])

    # Hybrid cache: some layers are attention layers with per-token KV.
    if prompt_len is not None and len(snapshot) < len(cache.layers):
        if hasattr(cache, "crop"):
            cache.crop(prompt_len)
        else:
            raise RuntimeError(
                "Hybrid model cache has attention layers holding previous-"
                "rollout KV entries, but this transformers version's "
                "DynamicCache has no crop() — cannot guarantee isolation. "
                "Upgrade transformers or use base_rollout for this model."
            )


def _compute_isolation_diagnostics(
    h0_clean: list[dict[str, torch.Tensor]],
    cache_post_rollout: DynamicCache,
) -> dict:
    """
    Compute diagnostics verifying that state isolation is working correctly.

    Returns dict with:
    - ssm_cosine_sim_mean: mean cosine similarity between h0 and post-rollout states
      (should be < 1.0 if the rollout changed states; ~1.0 means no generation happened)
    - ssm_l2_drift_mean: mean L2 distance of state drift from h0
    """
    cos_sims = []
    l2_drifts = []

    for h0 in h0_clean:
        h_post = cache_post_rollout.layers[h0["layer_idx"]].recurrent_states
        h0_flat = h0["recurrent"].flatten().float()
        h_post_flat = h_post.flatten().float()

        # Cosine similarity
        cos_sim = torch.nn.functional.cosine_similarity(
            h0_flat.unsqueeze(0), h_post_flat.unsqueeze(0)
        ).item()
        cos_sims.append(cos_sim)

        # L2 drift
        l2_drift = (h0_flat - h_post_flat).norm().item()
        l2_drifts.append(l2_drift)

    return {
        "sgrpo/ssm_cosine_sim_mean": sum(cos_sims) / len(cos_sims),
        "sgrpo/ssm_l2_drift_mean": sum(l2_drifts) / len(l2_drifts),
        "sgrpo/ssm_layers_checked": len(cos_sims),
    }


def generate_rollouts_isolated(
    model: MambaForCausalLM,
    tokenizer: AutoTokenizer,
    prompt: str,
    group_size: int = 4,
    max_new_tokens: int = 256,
    temperature: float = 1.0,
    device: str = "cuda",
) -> dict:
    """
    Generate group_size rollouts with SSM state isolation.
    Each rollout begins from an identical h_0 captured after prompt processing.

    Args:
        model:          the current policy model
        tokenizer:      tokenizer
        prompt:         formatted prompt string
        group_size:     G — number of rollouts to generate
        max_new_tokens: maximum tokens to generate per rollout
        temperature:    sampling temperature
        device:         "cuda" or "cpu"

    Returns same structure as base_rollout.generate_rollouts() for
    drop-in compatibility with the shared trainer, plus diagnostic fields.
    """
    model.eval()

    prompt_inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(device)

    prompt_len = prompt_inputs.input_ids.shape[1]

    # ── Step 1: Process prompt once, capture clean h_0 ───────────────────
    # The model auto-creates a DynamicCache when use_cache=True.
    # The final-position logits give the distribution over the FIRST
    # generated token — identical for every rollout since all start from h_0.
    isolation_start = time.time()
    with torch.no_grad():
        prompt_out = model(
            input_ids=prompt_inputs.input_ids,
            use_cache=True,
        )
        cache = prompt_out.cache_params
        h0_clean = _snapshot_ssm_states(cache)
        first_token_logits = prompt_out.logits[:, -1, :].detach().clone()
    isolation_time = time.time() - isolation_start

    # ── Step 2: Generate G rollouts, each from clean h_0 ────────────────
    all_generated_ids = []
    all_old_log_probs = []
    all_generated_texts = []
    isolation_diagnostics = None

    with torch.no_grad():
        for k in range(group_size):
            # Restore h_0 before every rollout — this is the isolation
            # mechanism. Pure tensor copy into the same cache object;
            # no prompt re-encoding needed. prompt_len lets hybrid caches
            # crop stale attention KV from the previous rollout.
            _restore_ssm_states(cache, h0_clean, prompt_len=prompt_len)

            generated_ids = prompt_inputs.input_ids.clone()
            log_probs = []
            logits = first_token_logits

            for t_idx in range(max_new_tokens):
                # Sample next token from current logits
                if temperature != 1.0:
                    step_logits = logits / temperature
                else:
                    step_logits = logits

                # Numerically stable log-softmax
                log_probs_dist = torch.log_softmax(step_logits, dim=-1)
                probs = torch.exp(log_probs_dist)
                next_token = torch.multinomial(probs, 1)  # [1, 1]

                # Store log_prob of sampled token
                log_prob = log_probs_dist[0, next_token[0, 0]]
                log_probs.append(log_prob)

                generated_ids = torch.cat([generated_ids, next_token], dim=1)

                # Stop at eos
                if next_token[0, 0].item() == tokenizer.eos_token_id:
                    break

                # Advance the state with the sampled token (decode step —
                # DynamicCache tracks position via has_previous_state)
                out = model(
                    input_ids=next_token,
                    cache_params=cache,
                    use_cache=True,
                )
                logits = out.logits[:, -1, :]

            # Compute isolation diagnostics on first rollout
            if k == 0:
                isolation_diagnostics = _compute_isolation_diagnostics(
                    h0_clean, cache
                )

            old_log_probs_tensor = torch.stack(log_probs)  # [gen_len]

            generated_text = tokenizer.decode(
                generated_ids[0, prompt_len:],
                skip_special_tokens=True
            )

            all_generated_ids.append(generated_ids[0])
            all_old_log_probs.append(old_log_probs_tensor)
            all_generated_texts.append(generated_text)

    # Pad log_probs to same length for stacking
    max_len = max(lp.shape[0] for lp in all_old_log_probs)
    padded_log_probs = []
    for lp in all_old_log_probs:
        pad_len = max_len - lp.shape[0]
        if pad_len > 0:
            padding = torch.zeros(pad_len, device=device, dtype=lp.dtype)
            lp = torch.cat([lp, padding])
        padded_log_probs.append(lp)

    result = {
        "input_ids": prompt_inputs.input_ids.expand(group_size, -1),
        "generated_ids": torch.nn.utils.rnn.pad_sequence(
            all_generated_ids, batch_first=True,
            padding_value=tokenizer.pad_token_id
        ),
        "generated_texts": all_generated_texts,
        "old_log_probs": torch.stack(padded_log_probs),  # [G, max_gen_len]
        "prompt_len": prompt_len,
        "h0_snapshot": h0_clean,               # kept for diagnostics
        "isolation_time": isolation_time,       # timing overhead
    }

    if isolation_diagnostics:
        result["isolation_diagnostics"] = isolation_diagnostics

    return result


In [ ]:
%%writefile trainer/__init__.py


In [ ]:
%%writefile trainer/base_trainer.py
"""
trainer/base_trainer.py

Shared training loop for all five algorithms.
The loss function and rollout generator are selected based on algorithm name.
Everything else is identical — optimizer, gradient clipping, logging, eval.

This is what makes the comparison scientifically valid:
one trainer, five loss functions, same everything else.

Research-grade features:
- Full metric computation per wandb_tracking_spec.md
- Evaluation loop on held-out test split
- Checkpoint saving with full state
- Histogram logging of key distributions
- Sample generation tables for qualitative inspection
- Timing instrumentation for throughput analysis
- Entropy and KL divergence computation
- Reference model for KL tracking
"""

import os
import re
import copy
import time
import random
import logging

import torch
from torch.optim import AdamW

from rewards.gsm8k_reward import compute_group_rewards
from tracking.wandb_logger import (
    init_run, init_local_sink, log_rollout, log_step, log_eval,
    log_histograms, log_sample_table, log_checkpoint,
    log_degenerate_group, log_run_metadata, finish,
)

# Import loss modules (not wildcard — each exports compute())
from losses import ppo_loss, grpo_loss, dapo_loss, bapo_loss, sgrpo_loss

# Import rollout generators
from rollouts.base_rollout import generate_rollouts
from rollouts.sgrpo_rollout import generate_rollouts_isolated

# Architecture detection is authoritative for rollout routing (see __init__).
from models.load_model import detect_architecture

logger = logging.getLogger(__name__)


class BaseTrainer:
    """
    Single trainer for all five algorithms.
    Algorithm-specific behavior is isolated to:
    1. Which rollout generator is used (base vs isolated)
    2. Which loss function is called
    Everything else is shared.
    """

    def __init__(self, model, tokenizer, train_dataset, eval_dataset,
                 algorithm: str, config, arch: str | None = None):
        self.model = model
        self.tokenizer = tokenizer
        self.train_dataset = train_dataset
        self.eval_dataset = eval_dataset
        self.algorithm = algorithm
        self.config = config

        # ── Architecture detection (authoritative) ───────────────────────
        # Rollout routing MUST be driven by the model's actual architecture,
        # never by a caller-supplied string or the algorithm name. A stale or
        # missing `arch` would route a stateless transformer into the SSM-only
        # isolated generator, whose _snapshot_ssm_states() raises when it finds
        # no recurrent state. Detecting here (before accelerator.prepare(), so
        # the module is still bare and unwrapped) makes every entry point
        # correct even if it never passed `arch`. The optional `arch` argument
        # is kept for backward compatibility but the detected value wins.
        detected_arch = detect_architecture(model)
        if arch is not None and arch != detected_arch:
            print(f"Note: caller passed arch={arch!r} but the model is "
                  f"{detected_arch!r} — using detected value for routing.")

        # Reward comes from the dataset (BaseRewardDataset interface) so the
        # trainer works on any verifiable benchmark. Plain datasets without
        # the interface fall back to the GSM8K reward for backward compat.
        self._train_reward_fn = getattr(
            train_dataset, "compute_group_rewards", compute_group_rewards
        )
        self._eval_reward_fn = getattr(
            eval_dataset, "compute_group_rewards", compute_group_rewards
        )
        self.arch = detected_arch
        self.step = 0

        # ── Optional multi-GPU via HuggingFace Accelerate ────────────────
        # When enabled (accelerate launch ... --use_accelerate), Accelerate
        # owns device placement, DDP wrapping, and mixed precision. When
        # disabled, everything below reduces to plain single-device PyTorch
        # so local runs are bit-identical to the pre-Accelerate trainer.
        if config.use_accelerate:
            from accelerate import Accelerator
            self.accelerator = Accelerator(
                mixed_precision=(
                    "bf16" if config.dtype == "bfloat16" else "no"
                ),
            )
            self.device = str(self.accelerator.device)
        else:
            self.accelerator = None
            self.device = config.device

        # Select rollout generator
        # SGRPO uses state-isolated rollouts — this is the novel contribution.
        # Isolation only applies to architectures that carry recurrent state
        # across generations (SSM and hybrid SSM+attention). Transformers are
        # stateless between generate() calls, so SGRPO degenerates to GRPO —
        # which is exactly the paper's control condition.
        if algorithm == "sgrpo" and self.arch in ("ssm", "hybrid"):
            self.rollout_fn = generate_rollouts_isolated
            self.arch_branch = "isolated"
            print(f"SGRPO: architecture={self.arch}, using isolated rollouts "
                  f"(SSM state isolation active).")
        elif algorithm == "sgrpo":
            self.rollout_fn = generate_rollouts
            self.arch_branch = "standard"
            print(f"SGRPO: architecture={self.arch}, using standard rollouts "
                  f"(state isolation not applicable — degenerates to GRPO, "
                  f"the paper's control condition).")
        else:
            self.rollout_fn = generate_rollouts
            self.arch_branch = "standard"

        # Optimizer — same for all algorithms
        self.optimizer = AdamW(
            model.parameters(),
            lr=config.lr,
            weight_decay=config.weight_decay,
        )

        # Frozen reference model for the KL penalty term.
        # Required by RLHF PPO (InstructGPT), GRPO (DeepSeekMath), and BAPO.
        # DAPO removes it by design; SGRPO replaces it with Future-KL
        # weighting computed from stored rollout log-probs (no second model).
        # Deepcopy happens BEFORE accelerator.prepare() so the reference is
        # the bare (non-DDP-wrapped) module.
        if algorithm in ("ppo", "grpo", "bapo"):
            self._ref_model = copy.deepcopy(model).to(self.device)
            self._ref_model.eval()
            for p in self._ref_model.parameters():
                p.requires_grad_(False)
            print(f"Reference model initialized for KL penalty "
                  f"(kl_coef={config.kl_coef})")
        else:
            self._ref_model = None

        # Under Accelerate the model comes back DDP-wrapped: gradient
        # forward/backward must go through self.model, but generation
        # (rollouts, eval) needs the bare module — DDP does not expose
        # generate() and adds no value inside torch.no_grad().
        if self.accelerator is not None:
            self.model, self.optimizer = self.accelerator.prepare(
                self.model, self.optimizer
            )
            self._gen_model = self.accelerator.unwrap_model(self.model)
        else:
            self._gen_model = self.model

        # PPO advantage baseline. PPO runs with group_size=1, so
        # group-relative statistics are undefined (std of one sample is
        # NaN). Without a value critic, the standard critic-free choice is
        # an exponential moving average of past rewards as the baseline:
        # A = r - EMA(r).
        self._reward_baseline: float | None = None

        # Tracking state
        self._all_rewards = []       # for histogram logging
        self._all_advantages = []
        self._all_ratios = []
        self._all_response_lengths = []
        self._sample_buffer = []     # for sample generation tables

        # Inner optimization epochs (PPO-style rollout-batch reuse). mu == 1
        # is the legacy single-pass path; mu > 1 steps the optimizer after
        # every inner epoch so theta moves between passes. See train().
        self._inner_epochs = max(1, int(getattr(config, "inner_epochs", 1)))
        self._global_update = 0      # monotonic optimizer-step counter

        # W&B init — main process only under multi-GPU
        if not config.no_wandb and self._is_main:
            init_run(
                algorithm=algorithm,
                run_name=config.run_name,
                config=config.to_dict(),
                project=config.wandb_project,
            )
        # Local JSONL metrics sink — always on (main process), so every run
        # (including --no_wandb smoke tests) leaves a structured metrics
        # record that Experiments/local_benchmark.py can graph offline.
        if self._is_main:
            local_path = init_local_sink(algorithm, config.run_name)
            print(f"Local metrics log: {local_path}")
            # One-time: record architecture + rollout branch (control vs
            # treatment) so both sinks capture the condition for each run.
            log_run_metadata(algorithm, self.arch, self.arch_branch)

        # Checkpoint dir
        os.makedirs(config.checkpoint_dir, exist_ok=True)

        if self._is_main:
            print(f"Trainer initialized: {algorithm.upper()} | arch: {self.arch}")
            print(f"Rollout generator: "
                  f"{'isolated (SGRPO)' if self.rollout_fn is generate_rollouts_isolated else 'standard'}")
            print(f"Inner epochs (mu): {self._inner_epochs}"
                  + ("" if self._inner_epochs > 1 else "  (legacy single-pass;"
                     " ratio/Future-KL are trivial at mu=1)"))
            if self.accelerator is not None:
                print(f"Accelerate: {self.accelerator.num_processes} "
                      f"process(es), device {self.device}, "
                      f"mixed_precision={self.accelerator.mixed_precision}")

    @property
    def _is_main(self) -> bool:
        """True on the main process (always True without Accelerate)."""
        return self.accelerator.is_main_process if self.accelerator else True

    def _zero_backward(self, anchor: torch.Tensor) -> None:
        """
        Backward a zero-valued surrogate through the existing forward graph.

        Under DDP every rank must run backward every microbatch, or the
        gradient all-reduce collectives desynchronize (hang) and the
        grad-accum counters drift apart (silent weight divergence). A rank
        that skips a step (degenerate group, non-finite loss) therefore
        contributes an explicit zero gradient instead of skipping backward.

        nan_to_num guards the non-finite-loss case: 0 * inf = NaN, so the
        anchor must be sanitized before the zero-multiply.
        """
        self.accelerator.backward(anchor.nan_to_num().sum() * 0.0)

    def _clip_grads(self) -> float:
        """Clip gradients, via Accelerate when active (handles DDP/AMP)."""
        if self.accelerator is not None:
            return self.accelerator.clip_grad_norm_(
                self.model.parameters(), self.config.max_grad_norm
            ).item()
        return torch.nn.utils.clip_grad_norm_(
            self.model.parameters(), max_norm=self.config.max_grad_norm
        ).item()

    def _step_if_boundary(self, accumulation_count: int) -> None:
        """Optimizer step at the grad-accum boundary (skip-path variant —
        the normal path steps inline where it also logs)."""
        if accumulation_count % self.config.grad_accum == 0:
            self._clip_grads()
            self.optimizer.step()
            self.optimizer.zero_grad()

    def _compute_new_log_probs_and_entropy(
        self,
        generated_ids: torch.Tensor,    # [G, full_len]
        prompt_len: int,
    ) -> tuple[torch.Tensor, float]:
        """
        Compute log probs under the CURRENT policy for generated sequences,
        plus policy entropy for collapse detection.

        Returns:
            new_log_probs: [G, gen_len]
            entropy:       scalar — average entropy across all generated positions
        """
        gen_len = generated_ids.shape[1] - prompt_len

        # Forward pass through current policy
        outputs = self.model(input_ids=generated_ids)
        logits = outputs.logits  # [G, full_len, vocab_size]

        # Log probs for generated tokens only
        gen_logits = logits[:, prompt_len - 1:-1, :]  # [G, gen_len, vocab_size]
        gen_ids = generated_ids[:, prompt_len:]        # [G, gen_len]

        log_probs = torch.log_softmax(gen_logits, dim=-1)

        # Gather log prob of the actual token at each position
        new_log_probs = log_probs.gather(
            2, gen_ids.unsqueeze(2)
        ).squeeze(2)  # [G, gen_len]

        # Compute entropy: H = -Σ p(x) log p(x)
        probs = torch.softmax(gen_logits, dim=-1)
        entropy = -(probs * log_probs).sum(dim=-1).mean().item()

        return new_log_probs, entropy

    def _compute_ref_log_probs(
        self,
        generated_ids: torch.Tensor,    # [G, full_len]
        prompt_len: int,
    ) -> torch.Tensor:
        """
        Compute log probs of the generated tokens under the FROZEN reference
        model. Used only by algorithms with a KL penalty (ppo/grpo/bapo).

        Returns:
            ref_log_probs: [G, gen_len], detached
        """
        with torch.no_grad():
            outputs = self._ref_model(input_ids=generated_ids)
            logits = outputs.logits  # [G, full_len, vocab_size]

            gen_logits = logits[:, prompt_len - 1:-1, :]
            gen_ids = generated_ids[:, prompt_len:]

            log_probs = torch.log_softmax(gen_logits, dim=-1)
            ref_log_probs = log_probs.gather(
                2, gen_ids.unsqueeze(2)
            ).squeeze(2)  # [G, gen_len]

        return ref_log_probs

    def _run_loss(
        self,
        new_log_probs: torch.Tensor,
        old_log_probs: torch.Tensor,
        rewards: torch.Tensor,
        attention_mask: torch.Tensor,
        ref_log_probs: torch.Tensor | None = None,
    ):
        """
        Call the correct loss function for the current algorithm.
        Returns (loss, advantages, metrics_dict) or None if degenerate group.

        All loss functions now return a metrics dict for WandB logging.
        """
        if self.algorithm == "ppo":
            # Critic-free baseline: EMA of past rewards (group stats are
            # undefined at group_size=1 — std of one sample is NaN)
            reward_mean = rewards.mean().item()
            if self._reward_baseline is None:
                self._reward_baseline = reward_mean
            advantages = rewards - self._reward_baseline
            self._reward_baseline = (
                0.9 * self._reward_baseline + 0.1 * reward_mean
            )
            loss, advantages, metrics = ppo_loss.compute(
                new_log_probs, old_log_probs, advantages, self.config.clip_eps,
                ref_log_probs=ref_log_probs, kl_coef=self.config.kl_coef,
            )
            metrics["ppo/reward_baseline"] = self._reward_baseline
            return loss, advantages, metrics

        elif self.algorithm == "grpo":
            result = grpo_loss.compute(
                new_log_probs, old_log_probs, rewards, self.config.clip_eps,
                ref_log_probs=ref_log_probs, kl_coef=self.config.kl_coef,
            )
            return result  # (loss, advantages, metrics)

        elif self.algorithm == "dapo":
            result = dapo_loss.compute(
                new_log_probs, old_log_probs, rewards,
                attention_mask, self.config.clip_eps
            )
            return result  # (loss, advantages, metrics) or None

        elif self.algorithm == "bapo":
            result = bapo_loss.compute(
                new_log_probs, old_log_probs, rewards, attention_mask,
                c_low_min=self.config.bapo_c_low_min,
                c_low_max=self.config.bapo_c_low_max,
                c_high_min=self.config.bapo_c_high_min,
                c_high_max=self.config.bapo_c_high_max,
                delta_high=self.config.bapo_delta_high,
                delta_low=self.config.bapo_delta_low,
                rho_target=self.config.bapo_rho_target,
                ref_log_probs=ref_log_probs, kl_coef=self.config.kl_coef,
            )
            return result  # (loss, advantages, metrics) or None

        elif self.algorithm == "sgrpo":
            result = sgrpo_loss.compute(
                new_log_probs, old_log_probs, rewards,
                attention_mask, self.config.clip_eps,
                self.config.future_kl_decay,
                self.config.future_kl_clip_low,
                self.config.future_kl_clip_high,
            )
            return result  # (loss, advantages, metrics) or None

    def _compute_response_stats(
        self,
        generated_texts: list[str],
        generated_ids: torch.Tensor,
        prompt_len: int,
    ) -> dict:
        """Compute response length and diversity statistics."""
        lengths = [len(text) for text in generated_texts]
        token_lengths = [
            (generated_ids[i, prompt_len:] != self.tokenizer.pad_token_id).sum().item()
            for i in range(generated_ids.shape[0])
        ]

        # Unique token ratio: vocabulary diversity metric
        all_gen_tokens = generated_ids[:, prompt_len:].flatten()
        real_tokens = all_gen_tokens[all_gen_tokens != self.tokenizer.pad_token_id]
        unique_ratio = (
            len(real_tokens.unique()) / max(len(real_tokens), 1)
        )

        return {
            "response_length_mean": sum(token_lengths) / max(len(token_lengths), 1),
            "response_length_std": (
                torch.tensor(token_lengths, dtype=torch.float32).std().item()
                if len(token_lengths) > 1 else 0.0
            ),
            "response_length_max": max(token_lengths) if token_lengths else 0,
            "response_length_min": min(token_lengths) if token_lengths else 0,
            "unique_tokens_ratio": unique_ratio,
            "raw_lengths": token_lengths,
        }

    def _evaluate(self, step: int) -> dict:
        """
        Run evaluation on the held-out test split.
        Computes accuracy, reward statistics, and response quality metrics.
        Returns the headline metrics so callers (benchmark runner) can
        aggregate across runs.
        """
        self.model.eval()
        eval_data = list(self.eval_dataset)
        random.shuffle(eval_data)
        eval_data = eval_data[:self.config.eval_samples]

        correct = 0
        total = 0
        all_rewards = []
        all_lengths = []
        reflection_count = 0
        self_correction_count = 0
        eval_samples = []

        with torch.no_grad():
            for item in eval_data:
                prompt = item["prompt"]
                answer = item["answer"]

                # Generate single greedy response for eval
                prompt_inputs = self.tokenizer(
                    prompt, return_tensors="pt", padding=True,
                    truncation=True, max_length=512,
                ).to(self.device)

                output = self._gen_model.generate(
                    prompt_inputs.input_ids,
                    attention_mask=prompt_inputs.attention_mask,
                    max_new_tokens=self.config.max_tokens,
                    do_sample=False,  # greedy for eval
                    pad_token_id=self.tokenizer.pad_token_id,
                )

                gen_text = self.tokenizer.decode(
                    output[0, prompt_inputs.input_ids.shape[1]:],
                    skip_special_tokens=True
                )

                reward = self._eval_reward_fn([gen_text], answer)[0]
                all_rewards.append(reward)
                correct += int(reward > 0.5)
                total += 1

                gen_len = output.shape[1] - prompt_inputs.input_ids.shape[1]
                all_lengths.append(gen_len)

                # Reasoning quality heuristics
                if re.search(r"\b(wait|alternatively|let me reconsider)\b",
                             gen_text, re.IGNORECASE):
                    reflection_count += 1
                if re.search(r"\b(actually|correction|I made an error)\b",
                             gen_text, re.IGNORECASE):
                    self_correction_count += 1

                # Buffer samples for table logging
                if len(eval_samples) < 10:
                    eval_samples.append({
                        "prompt": prompt,
                        "response": gen_text,
                        "reward": reward,
                        "response_length": gen_len,
                    })

        accuracy = correct / max(total, 1)
        avg_reward = sum(all_rewards) / max(len(all_rewards), 1)

        import numpy as np
        lengths_arr = np.array(all_lengths)

        if self._is_main:
            log_eval(
                step=step,
                algorithm=self.algorithm,
                gsm8k_accuracy=accuracy,
                average_reward=avg_reward,
                correct_count=correct,
                total_count=total,
                response_length_mean=float(lengths_arr.mean()),
                response_length_median=float(np.median(lengths_arr)),
                reasoning_steps_mean=None,
                reflection_count=reflection_count / max(total, 1),
                self_correction_rate=self_correction_count / max(total, 1),
            )

            # Log sample table
            if step % self.config.sample_table_every == 0:
                log_sample_table(step, eval_samples)

        print(f"  EVAL @ step {step}: accuracy={accuracy:.3f}, "
              f"avg_reward={avg_reward:.3f}, "
              f"correct={correct}/{total}")

        self.model.train()

        return {
            "step": step,
            "accuracy": accuracy,
            "average_reward": avg_reward,
            "correct": correct,
            "total": total,
            "response_length_mean": float(lengths_arr.mean()),
        }

    def _save_checkpoint(self, step: int) -> str:
        """Save model checkpoint and return the path.

        The filename includes run_name so benchmark runs of the same
        algorithm with different seeds don't overwrite each other.
        Optimizer state is optional (config.save_optimizer_state) — the
        benchmark disables it because its checkpoints exist for test-set
        evaluation, not resuming, and AdamW state triples the file size.
        """
        path = os.path.join(
            self.config.checkpoint_dir,
            f"{self.algorithm}_{self.config.run_name}_step{step}.pt"
        )
        checkpoint = {
            "step": step,
            "algorithm": self.algorithm,
            # Bare-module weights — loadable without DDP/Accelerate
            "model_state_dict": self._gen_model.state_dict(),
            "config": self.config.to_dict(),
        }
        if getattr(self.config, "save_optimizer_state", True):
            checkpoint["optimizer_state_dict"] = self.optimizer.state_dict()
        torch.save(checkpoint, path)
        print(f"  Checkpoint saved: {path}")
        return path

    def train(self):
        """Main training loop with full research-grade instrumentation."""
        self.model.train()
        data = list(self.train_dataset)
        random.shuffle(data)

        accumulation_count = 0
        self.optimizer.zero_grad()

        # Under multi-GPU each process trains on a disjoint slice of the
        # shuffled prompt stream (stride = world size); DDP averages the
        # gradients, so one optimizer step consumes world_size prompts.
        rank = self.accelerator.process_index if self.accelerator else 0
        world = self.accelerator.num_processes if self.accelerator else 1

        for step in range(self.config.steps):
            step_start_time = time.time()
            # Track progress even when a step is skipped (degenerate group,
            # non-finite loss) so the final report reflects steps attempted.
            self.step = step

            # Sample a prompt
            item = data[(step * world + rank) % len(data)]
            prompt = item["prompt"]
            answer = item["answer"]

            # ── Rollout phase ────────────────────────────────────────────
            gen_start_time = time.time()
            self.model.eval()
            with torch.no_grad():
                rollout_data = self.rollout_fn(
                    # Bare module: rollout generators call generate()/custom
                    # decode loops, which DDP wrappers don't expose. Shares
                    # parameters with self.model, so it's the current policy.
                    model=self._gen_model,
                    tokenizer=self.tokenizer,
                    prompt=prompt,
                    group_size=self.config.group_size,
                    max_new_tokens=self.config.max_tokens,
                    temperature=self.config.temperature,
                    device=self.device,
                )
            generation_time = time.time() - gen_start_time

            # Compute rewards (via the dataset's verifiable reward)
            rewards_list = self._train_reward_fn(
                rollout_data["generated_texts"], answer
            )
            rewards = torch.tensor(
                rewards_list, dtype=torch.float32, device=self.device
            )

            # ── Response statistics ──────────────────────────────────────
            # Computed BEFORE the degenerate check (cheap token counting) so
            # rollout-phase metrics are logged for every step, including the
            # skipped ones.
            generated_ids = rollout_data["generated_ids"].to(self.device)
            prompt_len = rollout_data["prompt_len"]
            gen_len = generated_ids.shape[1] - prompt_len

            response_stats = self._compute_response_stats(
                rollout_data["generated_texts"], generated_ids, prompt_len
            )

            is_degenerate = (
                self.algorithm in ("dapo", "bapo", "sgrpo")
                and dapo_loss.is_degenerate_group(rewards)
            )

            # ── Rollout-phase RL metrics — logged EVERY step ─────────────
            # log_step only fires on optimizer updates; early in training
            # nearly every group is degenerate, so without this the W&B run
            # shows little beyond GPU/system charts. rollout/* keeps reward
            # signal, response lengths, and the degenerate flag visible
            # from step 0.
            if self._is_main:
                log_rollout(
                    step=step,
                    algorithm=self.algorithm,
                    reward_mean=rewards.mean().item(),
                    reward_std=rewards.std().item() if rewards.numel() > 1 else 0.0,
                    reward_max=rewards.max().item(),
                    reward_min=rewards.min().item(),
                    reward_positive_fraction=(rewards > 0).float().mean().item(),
                    response_length_mean=response_stats["response_length_mean"],
                    response_length_std=response_stats["response_length_std"],
                    response_length_max=response_stats["response_length_max"],
                    response_length_min=response_stats["response_length_min"],
                    unique_tokens_ratio=response_stats["unique_tokens_ratio"],
                    generation_time=generation_time,
                    group_size=self.config.group_size,
                    is_degenerate=is_degenerate,
                )

            # ── Early degenerate-group check ─────────────────────────────
            # DAPO/BAPO/SGRPO skip groups where all rewards are identical
            # (advantages would be zero). Checking here — before the
            # gradient forward pass — matters for two reasons:
            #   1. It saves an entire wasted forward over [G, full_len].
            #   2. Skipping AFTER the forward leaks the autograd graph:
            #      with no backward() to free it, the graph stays alive
            #      through the next step's forward, doubling peak VRAM
            #      (observed OOM on 6 GB with Mamba's slow path).
            # Under Accelerate this early exit is disabled: skipping before
            # the forward would leave this rank with no backward to
            # contribute, desynchronizing DDP. The degenerate group instead
            # falls through to _run_loss -> None, where _zero_backward keeps
            # the collectives in lockstep. (Multi-GPU nodes have the VRAM
            # headroom the early exit exists to save.)
            if self.accelerator is None and is_degenerate:
                if self._is_main:
                    log_degenerate_group(step)
                if step % 10 == 0:
                    print(f"Step {step}: degenerate group "
                          f"(all same reward), skipping.")
                continue

            # ── Optimization phase ───────────────────────────────────────
            # Inner optimization epochs (PPO-style rollout-batch reuse):
            #
            #   rollouts are generated ONCE per outer step above (old_log_probs
            #   fixed at rollout-time theta). We then re-optimize that same
            #   batch `mu` times. old_log_probs stays fixed across all mu inner
            #   passes; only new_log_probs is recomputed each pass because
            #   theta has moved. At inner_epoch 0 theta == theta_old, so
            #   ratio == 1 and Future-KL == 0 (identical to the single-pass
            #   trainer); from inner_epoch 1 onward theta has been stepped, so
            #   the importance ratio and Future-KL become non-trivial — the
            #   entire point of Fix 3.
            #
            # Composition with grad_accum (they are orthogonal in intent but
            # both gate the optimizer step, so the reconciliation is explicit,
            # never silent):
            #   - mu == 1 (default): LEGACY path. Gradients accumulate across
            #     `grad_accum` consecutive PROMPTS; one optimizer step per
            #     grad_accum prompts. Bit-identical to the pre-Fix-3 trainer.
            #   - mu > 1: theta MUST move between inner epochs or the reuse is
            #     pointless (new_log_probs would equal old_log_probs), so the
            #     optimizer steps after EVERY inner epoch and grad_accum's
            #     cross-prompt accumulation is not applied (each pass divides by
            #     1 and steps). The step count is visible in the console/W&B via
            #     sys/global_update and sys/inner_epoch.
            self.model.train()

            old_log_probs = rollout_data["old_log_probs"].to(self.device)

            # Build attention mask for generated tokens
            attention_mask = (
                generated_ids[:, prompt_len:] != self.tokenizer.pad_token_id
            ).float()

            # Align old_log_probs length with gen_len
            if old_log_probs.shape[1] > gen_len:
                old_log_probs = old_log_probs[:, :gen_len]
            elif old_log_probs.shape[1] < gen_len:
                pad = torch.zeros(
                    old_log_probs.shape[0],
                    gen_len - old_log_probs.shape[1],
                    device=self.device,
                )
                old_log_probs = torch.cat([old_log_probs, pad], dim=1)

            # Reference log probs for KL penalty (ppo/grpo/bapo only). The
            # reference model is frozen, so ref_log_probs are theta-independent
            # — compute ONCE and reuse across all inner epochs.
            ref_log_probs = (
                self._compute_ref_log_probs(generated_ids, prompt_len)
                if self._ref_model is not None else None
            )

            mu = self._inner_epochs
            step_every_pass = mu > 1
            loss_divisor = 1.0 if step_every_pass else self.config.grad_accum

            for inner_epoch in range(mu):
                # Forward pass under the CURRENT policy (theta may have been
                # stepped by a previous inner epoch).
                new_log_probs, entropy = self._compute_new_log_probs_and_entropy(
                    generated_ids, prompt_len
                )

                result = self._run_loss(
                    new_log_probs, old_log_probs, rewards, attention_mask,
                    ref_log_probs=ref_log_probs,
                )

                # ── Skip paths: degenerate group or non-finite loss ──────
                # Single-GPU: dapo/bapo/sgrpo degenerate groups are already
                # filtered before this loop, so `result is None` here only
                # happens under Accelerate (early exit disabled to keep DDP
                # collectives aligned). Non-finite losses are guarded for all.
                bad_reason = None
                if result is None:
                    bad_reason = "degenerate"
                elif not torch.isfinite(result[0]):
                    bad_reason = "non-finite"

                if bad_reason is not None:
                    if self.accelerator is not None:
                        # Zero contribution keeps DDP in lockstep; every rank
                        # runs exactly mu passes with an identical step cadence.
                        self._zero_backward(new_log_probs)
                        accumulation_count += 1
                        if step_every_pass or (
                            accumulation_count % self.config.grad_accum == 0
                        ):
                            self._clip_grads()
                            self.optimizer.step()
                            self.optimizer.zero_grad()
                            self._global_update += 1
                    else:
                        # Single-GPU: free the graph, take no step. For mu == 1
                        # this reproduces the legacy non-finite reset exactly.
                        self.optimizer.zero_grad()
                        if not step_every_pass:
                            accumulation_count = 0
                    if bad_reason == "degenerate":
                        if self._is_main:
                            log_degenerate_group(step)
                        if step % 10 == 0 and self._is_main:
                            print(f"Step {step}: degenerate group "
                                  f"(all same reward), skipping.")
                    elif self._is_main:
                        print(f"Step {step} (inner {inner_epoch}): non-finite "
                              f"loss, skipping update.")
                    del new_log_probs, entropy
                    if result is not None:
                        del result
                    continue

                loss, advantages, algo_metrics = result

                # ── Accumulate tracking data for histograms ──────────────
                with torch.no_grad():
                    ratio = torch.exp(
                        (new_log_probs - old_log_probs).clamp(-20.0, 20.0)
                    )
                    self._all_rewards.append(rewards.detach().cpu())
                    self._all_advantages.append(advantages.detach().cpu())
                    self._all_ratios.append(ratio.detach().cpu().flatten())
                    self._all_response_lengths.extend(
                        response_stats["raw_lengths"]
                    )

                # Scale loss (grad_accum for the legacy mu==1 path; 1 when
                # stepping every inner pass).
                scaled_loss = loss / loss_divisor
                if self.accelerator is not None:
                    self.accelerator.backward(scaled_loss)
                else:
                    scaled_loss.backward()

                accumulation_count += 1
                do_step = step_every_pass or (
                    accumulation_count % self.config.grad_accum == 0
                )

                if do_step:
                    # Gradient clipping
                    grad_norm = self._clip_grads()
                    grad_was_clipped = grad_norm > self.config.max_grad_norm

                    self.optimizer.step()
                    self.optimizer.zero_grad()
                    self._global_update += 1

                    step_time = time.time() - step_start_time

                    # Throughput: tokens processed per second
                    total_tokens = (
                        self.config.group_size * gen_len
                    )
                    throughput = total_tokens / max(step_time, 1e-6)

                    # Per-inner-epoch diagnostics ride along in the algo-metric
                    # passthrough so Future-KL's signal (zero on inner_epoch 0,
                    # nonzero afterwards) is visible in W&B.
                    passthrough = {
                        k: v for k, v in algo_metrics.items()
                        if not k.startswith("train/")
                    }
                    passthrough["sys/inner_epoch"] = inner_epoch
                    passthrough["sys/global_update"] = self._global_update

                    # ── Metrics logging: W&B + local JSONL (main only) ───
                    if self._is_main:
                        log_step(
                            step=step,
                            algorithm=self.algorithm,
                            loss=loss.item(),
                            policy_loss=algo_metrics.get(
                                f"{self.algorithm}/policy_loss", loss.item()
                            ),
                            kl_loss=algo_metrics.get(
                                f"{self.algorithm}/kl_penalty"
                            ),
                            entropy=entropy,
                            clip_fraction=algo_metrics.get("train/clip_fraction"),
                            approx_kl=algo_metrics.get("train/approx_kl"),
                            ratio_mean=algo_metrics.get("train/ratio_mean"),
                            ratio_std=algo_metrics.get("train/ratio_std"),
                            ratio_max=algo_metrics.get("train/ratio_max"),
                            ratio_min=algo_metrics.get("train/ratio_min"),
                            reward_mean=rewards.mean().item(),
                            reward_std=rewards.std().item(),
                            reward_max=rewards.max().item(),
                            reward_min=rewards.min().item(),
                            advantage_mean=advantages.mean().item(),
                            advantage_std=advantages.std().item(),
                            advantage_max=advantages.max().item(),
                            advantage_min=advantages.min().item(),
                            positive_advantage_ratio=algo_metrics.get(
                                "train/positive_advantage_ratio"
                            ),
                            grad_norm=grad_norm,
                            grad_was_clipped=grad_was_clipped,
                            response_length_mean=response_stats["response_length_mean"],
                            response_length_std=response_stats["response_length_std"],
                            response_length_max=response_stats["response_length_max"],
                            response_length_min=response_stats["response_length_min"],
                            unique_tokens_ratio=response_stats["unique_tokens_ratio"],
                            step_time=step_time,
                            generation_time=generation_time,
                            throughput=throughput,
                            algo_metrics=passthrough,
                        )

                        # Histogram logging
                        if step % self.config.histogram_every == 0 and step > 0:
                            if self._all_advantages:
                                log_histograms(
                                    step=step,
                                    advantages=torch.cat(self._all_advantages),
                                    rewards=torch.cat(self._all_rewards),
                                    ratios=torch.cat(self._all_ratios),
                                    response_lengths=self._all_response_lengths.copy(),
                                )
                                # Reset buffers
                                self._all_rewards.clear()
                                self._all_advantages.clear()
                                self._all_ratios.clear()
                                self._all_response_lengths.clear()

                    # ── Console output ───────────────────────────────────
                    if step % 10 == 0 and self._is_main:
                        if mu > 1:
                            # Richer line surfaces the per-inner-epoch signal
                            # that Fix 3 activates.
                            clipf = algo_metrics.get("train/clip_fraction")
                            clip_tag = (f" | ClipFrac: {clipf:.3f}"
                                        if clipf is not None else "")
                            sgrpo_tag = ""
                            if self.algorithm == "sgrpo":
                                sgrpo_tag = (
                                    f" | w_std: "
                                    f"{algo_metrics.get('sgrpo/influence_weight_std', 0.0):.2e}"
                                )
                            print(
                                f"Step {step:4d} [inner {inner_epoch+1}/{mu}] "
                                f"| Loss: {loss.item():.4f} "
                                f"| Reward: {rewards.mean().item():.3f} "
                                f"| Adv: {advantages.mean().item():.3f} "
                                f"| Entropy: {entropy:.2f} "
                                f"| GradNorm: {grad_norm:.3f}"
                                f"{clip_tag}{sgrpo_tag} "
                                f"| {step_time:.1f}s"
                            )
                        else:
                            # Legacy single-pass console line (unchanged).
                            print(
                                f"Step {step:4d} | Loss: {loss.item():.4f} "
                                f"| Reward: {rewards.mean().item():.3f} "
                                f"| Adv: {advantages.mean().item():.3f} "
                                f"| Entropy: {entropy:.2f} "
                                f"| GradNorm: {grad_norm:.3f} "
                                f"| {step_time:.1f}s"
                            )

                # Free this inner pass's forward graph before the next pass /
                # next step so peak VRAM does not grow with mu.
                del new_log_probs, entropy, loss, result

            # ── Periodic evaluation (main process only) ──────────────────
            if (step > 0 and step % self.config.eval_every == 0
                    and self._is_main):
                self._evaluate(step)

            # ── Periodic checkpoint (main process only) ──────────────────
            if (step > 0 and step % self.config.checkpoint_every == 0
                    and self._is_main):
                ckpt_path = self._save_checkpoint(step)
                if not self.config.no_wandb:
                    log_checkpoint(step, self.algorithm, ckpt_path)

        # ── Final eval and checkpoint (main process only) ─────────────────
        final_eval = None
        final_ckpt_path = None
        if self.accelerator is not None:
            self.accelerator.wait_for_everyone()
        if self._is_main:
            print(f"\nTraining complete. Final step: {self.step}")
            final_eval = self._evaluate(self.step)
            final_ckpt_path = self._save_checkpoint(self.step)
            if not self.config.no_wandb:
                log_checkpoint(self.step, self.algorithm, final_ckpt_path)
            finish()

        return {
            "algorithm": self.algorithm,
            "steps_attempted": self.step + 1 if self.config.steps else 0,
            "final_eval": final_eval,   # None on non-main ranks
            "final_checkpoint": final_ckpt_path,   # None on non-main ranks
        }


In [ ]:
%%writefile tracking/__init__.py


In [ ]:
%%writefile tracking/wandb_logger.py
"""
tracking/wandb_logger.py

Research-grade Weights & Biases logging for the SGRPO project.

Implements EVERY metric from Docs/wandb_tracking_spec.md:
- Universal metrics (every step): loss, entropy, KL, ratios, rewards, advantages
- Algorithm-specific metrics (every step): per-algorithm diagnostics
- Gradient health (every 10 steps): grad norms, clip counts
- Histograms (every 50 steps): distributions of key quantities
- Evaluation metrics (every eval_every steps): accuracy, response quality
- System metrics (every step): GPU memory, throughput, timing
- Artifacts: model checkpoints, code snapshots, sample generation tables

RULE: Never call wandb.log() directly from any other module.
All logging goes through this module's functions.
"""

import os
import json
import time
import logging
from typing import Optional
from dataclasses import asdict

import torch
import wandb

logger = logging.getLogger(__name__)


# ── Module state ─────────────────────────────────────────────────────────────
_run_active = False
_grad_clip_count = 0
_degenerate_group_count = 0
_total_groups = 0

# Local JSONL metrics sink. Independent of W&B: active for --no_wandb runs
# (where it is the only record) and alongside W&B runs. One JSON object per
# line, tagged with a "kind" field (metadata / step / degenerate / eval).
# Experiments/local_benchmark.py consumes these files to build offline
# comparison graphs, so every algorithm writes the same schema here that it
# sends to W&B.
_local_sink = None
_local_sink_path = None


def init_local_sink(
    algorithm: str,
    run_name: str,
    out_dir: Optional[str] = None,
) -> str:
    """
    Open a local JSONL metrics file for this run.

    out_dir precedence: explicit arg > SGRPO_LOCAL_LOG_DIR env var (set by
    Experiments/local_benchmark.py to group one benchmark's runs together) >
    Experiments/results/local_logs.
    """
    global _local_sink, _local_sink_path
    if out_dir is None:
        out_dir = os.environ.get(
            "SGRPO_LOCAL_LOG_DIR",
            os.path.join("Experiments", "results", "local_logs"),
        )
    os.makedirs(out_dir, exist_ok=True)
    stamp = time.strftime("%Y%m%d_%H%M%S")
    _local_sink_path = os.path.join(
        out_dir, f"{algorithm}_{run_name}_{stamp}.jsonl"
    )
    _local_sink = open(_local_sink_path, "a", encoding="utf-8")
    return _local_sink_path


def _local_log(kind: str, payload: dict) -> None:
    """Append one record to the local sink, keeping JSON-safe scalars only."""
    if _local_sink is None:
        return
    record = {"kind": kind, "wall_time": time.time()}
    for k, v in payload.items():
        if v is None or isinstance(v, (int, float, str, bool)):
            record[k] = v
    _local_sink.write(json.dumps(record) + "\n")
    _local_sink.flush()


def init_run(
    algorithm: str,
    run_name: str,
    config: dict,
    project: str = "SGRPO",
) -> None:
    """
    Initialize a W&B run with the full config from wandb_tracking_spec.md.

    Config includes all hyperparameters, environment info, hardware info,
    and git commit hash for full reproducibility.
    """
    global _run_active, _grad_clip_count, _degenerate_group_count, _total_groups
    _grad_clip_count = 0
    _degenerate_group_count = 0
    _total_groups = 0

    wandb.init(
        project=project,
        name=f"{algorithm}_{run_name}",
        tags=[algorithm, "mamba-130m", "gsm8k", "math-rl", "comparison"],
        config=config,
        save_code=True,
    )

    # Define custom x-axis for clarity
    wandb.define_metric("train/*", step_metric="sys/step")
    wandb.define_metric("rollout/*", step_metric="sys/step")
    wandb.define_metric("eval/*", step_metric="sys/step")
    wandb.define_metric("system/*", step_metric="sys/step")
    wandb.define_metric("hist/*", step_metric="sys/step")

    # Save all Python source files as code artifact
    try:
        _save_code_snapshot()
    except Exception as e:
        logger.warning(f"Failed to save code snapshot: {e}")

    _run_active = True
    logger.info(f"W&B run initialized: {algorithm}_{run_name} in project {project}")


def _save_code_snapshot():
    """Save all .py files as a W&B artifact for reproducibility."""
    artifact = wandb.Artifact("source_code", type="code")
    project_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    for root, dirs, files in os.walk(project_root):
        # Skip hidden dirs, __pycache__, .venv, etc.
        dirs[:] = [d for d in dirs if not d.startswith(('.', '__'))
                   and d not in ('venv', '.venv', 'node_modules', '.git')]
        for f in files:
            if f.endswith('.py'):
                filepath = os.path.join(root, f)
                relpath = os.path.relpath(filepath, project_root)
                artifact.add_file(filepath, name=relpath)
    wandb.log_artifact(artifact)


def log_rollout(
    step: int,
    algorithm: str,
    # ── Reward statistics (from the rollout group) ───────────────────────
    reward_mean: float,
    reward_std: float,
    reward_max: float,
    reward_min: float,
    reward_positive_fraction: Optional[float] = None,
    # ── Response statistics ──────────────────────────────────────────────
    response_length_mean: Optional[float] = None,
    response_length_std: Optional[float] = None,
    response_length_max: Optional[float] = None,
    response_length_min: Optional[float] = None,
    unique_tokens_ratio: Optional[float] = None,
    # ── Rollout phase timing / meta ──────────────────────────────────────
    generation_time: Optional[float] = None,
    group_size: Optional[int] = None,
    is_degenerate: bool = False,
) -> None:
    """
    Log rollout-phase RL metrics for EVERY training step — including steps
    that are subsequently skipped as degenerate groups.

    log_step() only fires when an optimizer update happens; early in
    training (untrained model, all-zero rewards) nearly every dapo/bapo/
    sgrpo step is degenerate, so without this function the W&B run shows
    almost nothing but system/GPU metrics. rollout/* charts fill that gap:
    reward signal, response lengths, and degenerate rate are visible from
    step 0 regardless of whether the optimization phase ran.
    """
    if not _run_active and _local_sink is None:
        return

    payload = {
        "sys/step": step,
        "sys/algorithm": algorithm,
        "rollout/reward_mean": reward_mean,
        "rollout/reward_std": reward_std,
        "rollout/reward_max": reward_max,
        "rollout/reward_min": reward_min,
        "rollout/reward_positive_fraction": reward_positive_fraction,
        "rollout/response_length_mean": response_length_mean,
        "rollout/response_length_std": response_length_std,
        "rollout/response_length_max": response_length_max,
        "rollout/response_length_min": response_length_min,
        "rollout/unique_tokens_ratio": unique_tokens_ratio,
        "rollout/generation_time": generation_time,
        "rollout/group_size": group_size,
        "rollout/is_degenerate": int(is_degenerate),
    }
    payload = {k: v for k, v in payload.items() if v is not None}

    _local_log("rollout", payload)
    if _run_active:
        wandb.log(payload)


def log_step(
    step: int,
    algorithm: str,
    # ── Core training metrics ────────────────────────────────────────────
    loss: float,
    policy_loss: Optional[float] = None,
    kl_loss: Optional[float] = None,
    entropy: Optional[float] = None,
    # ── Clipping / ratio statistics ──────────────────────────────────────
    clip_fraction: Optional[float] = None,
    approx_kl: Optional[float] = None,
    ratio_mean: Optional[float] = None,
    ratio_std: Optional[float] = None,
    ratio_max: Optional[float] = None,
    ratio_min: Optional[float] = None,
    # ── Reward statistics ────────────────────────────────────────────────
    reward_mean: float = 0.0,
    reward_std: float = 0.0,
    reward_max: Optional[float] = None,
    reward_min: Optional[float] = None,
    # ── Advantage statistics ─────────────────────────────────────────────
    advantage_mean: float = 0.0,
    advantage_std: float = 0.0,
    advantage_max: Optional[float] = None,
    advantage_min: Optional[float] = None,
    positive_advantage_ratio: Optional[float] = None,
    # ── Gradient health ──────────────────────────────────────────────────
    grad_norm: Optional[float] = None,
    grad_norm_policy: Optional[float] = None,
    grad_was_clipped: bool = False,
    # ── Response statistics ──────────────────────────────────────────────
    response_length_mean: Optional[float] = None,
    response_length_std: Optional[float] = None,
    response_length_max: Optional[float] = None,
    response_length_min: Optional[float] = None,
    unique_tokens_ratio: Optional[float] = None,
    # ── Timing ───────────────────────────────────────────────────────────
    step_time: Optional[float] = None,
    generation_time: Optional[float] = None,
    throughput: Optional[float] = None,
    # ── Algorithm-specific extras ────────────────────────────────────────
    algo_metrics: Optional[dict] = None,
) -> None:
    """
    Log one training step. Every algorithm calls this with the same schema.

    Optional fields are None for algorithms that don't compute them.
    The WandB column still exists for cross-run comparison alignment.
    """
    global _grad_clip_count, _total_groups
    if not _run_active and _local_sink is None:
        return

    _total_groups += 1
    if grad_was_clipped:
        _grad_clip_count += 1

    # ── Build payload ────────────────────────────────────────────────────
    payload = {
        "sys/step": step,
        "sys/algorithm": algorithm,
        # Training dynamics
        "train/loss": loss,
        "train/policy_loss": policy_loss,
        "train/kl_loss": kl_loss,
        "train/entropy": entropy,
        "train/clip_fraction": clip_fraction,
        "train/approx_kl": approx_kl,
        "train/ratio_mean": ratio_mean,
        "train/ratio_std": ratio_std,
        "train/ratio_max": ratio_max,
        "train/ratio_min": ratio_min,
        # Rewards
        "train/reward_mean": reward_mean,
        "train/reward_std": reward_std,
        "train/reward_max": reward_max,
        "train/reward_min": reward_min,
        # Advantages
        "train/advantage_mean": advantage_mean,
        "train/advantage_std": advantage_std,
        "train/advantage_max": advantage_max,
        "train/advantage_min": advantage_min,
        "train/positive_advantage_ratio": positive_advantage_ratio,
        # Gradients
        "train/grad_norm": grad_norm,
        "train/grad_norm_policy": grad_norm_policy,
        "train/grad_clip_count": _grad_clip_count,
        # Response statistics
        "train/response_length_mean": response_length_mean,
        "train/response_length_std": response_length_std,
        "train/response_length_max": response_length_max,
        "train/response_length_min": response_length_min,
        "train/unique_tokens_ratio": unique_tokens_ratio,
        # Timing
        "system/step_time": step_time,
        "system/generation_time": generation_time,
        "system/throughput": throughput,
    }

    # ── System metrics ───────────────────────────────────────────────────
    if torch.cuda.is_available():
        payload["system/gpu_memory_allocated"] = (
            torch.cuda.memory_allocated() / 1e9
        )
        payload["system/gpu_memory_reserved"] = (
            torch.cuda.memory_reserved() / 1e9
        )

    # ── Algorithm-specific metrics ───────────────────────────────────────
    if algo_metrics:
        payload.update(algo_metrics)

    # Remove None values to keep WandB clean but preserve schema
    payload = {k: v for k, v in payload.items() if v is not None}

    _local_log("step", payload)
    if _run_active:
        # No explicit step= — every chart uses sys/step via define_metric.
        # Mixing explicit steps with step-less calls (log_run_metadata) makes
        # wandb silently drop any log whose step is behind its internal
        # counter, which is how early-run RL metrics went missing.
        wandb.log(payload)


def log_run_metadata(algorithm: str, arch: str, arch_branch: str) -> None:
    """
    Log one-time run metadata at trainer start.

    Records the detected model architecture and which rollout branch the run
    took, so W&B captures the control-vs-treatment condition per run:
      - arch_branch == "isolated": SGRPO on an SSM/hybrid model — state
        isolation active (the treatment).
      - arch_branch == "standard": SGRPO on a stateless transformer —
        isolation is a no-op, degenerates to GRPO (the paper's control), OR
        any non-SGRPO algorithm.

    Logged once (not per step), so it uses W&B's default step counter.
    """
    payload = {
        "sys/architecture": arch,
        "sys/architecture_branch": arch_branch,
        "sys/algorithm": algorithm,
    }
    _local_log("metadata", payload)
    if not _run_active:
        return
    wandb.log({
        "sys/architecture": arch,
        "sys/architecture_branch": arch_branch,
    })


def log_degenerate_group(step: int):
    """Track when a group is skipped due to degenerate rewards (all same)."""
    global _degenerate_group_count, _total_groups
    _degenerate_group_count += 1
    _total_groups += 1

    payload = {
        "train/degenerate_group_rate": _degenerate_group_count / max(_total_groups, 1),
        "sys/step": step,
    }
    _local_log("degenerate", payload)
    if _run_active:
        wandb.log(payload)


def log_histograms(
    step: int,
    advantages: Optional[torch.Tensor] = None,
    rewards: Optional[torch.Tensor] = None,
    ratios: Optional[torch.Tensor] = None,
    response_lengths: Optional[list] = None,
    token_probs: Optional[torch.Tensor] = None,
    gradients: Optional[dict] = None,
) -> None:
    """
    Log distribution histograms every histogram_every steps.
    These are critical for detecting:
    - advantage collapse (all near zero → degenerate training)
    - ratio explosion (extreme importance sampling weights)
    - response length mode collapse
    """
    if not _run_active:
        return

    payload = {"sys/step": step}

    if advantages is not None:
        payload["hist/advantages"] = wandb.Histogram(
            advantages.detach().cpu().float().numpy()
        )
    if rewards is not None:
        payload["hist/rewards"] = wandb.Histogram(
            rewards.detach().cpu().float().numpy()
        )
    if ratios is not None:
        payload["hist/ratios"] = wandb.Histogram(
            ratios.detach().cpu().float().numpy().clip(-10, 10)
        )
    if response_lengths is not None:
        payload["hist/response_lengths"] = wandb.Histogram(response_lengths)
    if token_probs is not None:
        payload["hist/token_probs"] = wandb.Histogram(
            token_probs.detach().cpu().float().numpy()
        )

    wandb.log(payload)


def log_eval(
    step: int,
    algorithm: str,
    # ── Task performance ─────────────────────────────────────────────────
    gsm8k_accuracy: Optional[float] = None,
    average_reward: Optional[float] = None,
    correct_count: Optional[int] = None,
    total_count: Optional[int] = None,
    # ── Reasoning quality ────────────────────────────────────────────────
    response_length_mean: Optional[float] = None,
    response_length_median: Optional[float] = None,
    reasoning_steps_mean: Optional[float] = None,
    reflection_count: Optional[float] = None,
    self_correction_rate: Optional[float] = None,
    # ── Distribution analysis ────────────────────────────────────────────
    entropy: Optional[float] = None,
    kl_from_ref: Optional[float] = None,
) -> None:
    """Log evaluation metrics on held-out test set."""
    if not _run_active and _local_sink is None:
        return

    payload = {
        "sys/step": step,
        "sys/algorithm": algorithm,
        "eval/gsm8k_accuracy": gsm8k_accuracy,
        "eval/average_reward": average_reward,
        "eval/correct_count": correct_count,
        "eval/total_count": total_count,
        "eval/response_length_mean": response_length_mean,
        "eval/response_length_median": response_length_median,
        "eval/reasoning_steps_mean": reasoning_steps_mean,
        "eval/reflection_count": reflection_count,
        "eval/self_correction_rate": self_correction_rate,
        "eval/entropy": entropy,
        "eval/kl_from_ref": kl_from_ref,
    }

    payload = {k: v for k, v in payload.items() if v is not None}
    _local_log("eval", payload)
    if _run_active:
        wandb.log(payload)


def log_sample_table(
    step: int,
    samples: list[dict],
) -> None:
    """
    Log a W&B table of sample generations for qualitative inspection.

    Each sample dict should have: prompt, response, reward, response_length
    """
    if not _run_active:
        return

    columns = ["step", "prompt", "response", "reward", "response_length"]
    table = wandb.Table(columns=columns)

    for s in samples:
        table.add_data(
            step,
            s.get("prompt", "")[:500],   # truncate for readability
            s.get("response", "")[:1000],
            s.get("reward", 0.0),
            s.get("response_length", 0),
        )

    wandb.log({"eval/samples": table, "sys/step": step})


def log_checkpoint(
    step: int,
    algorithm: str,
    checkpoint_path: str,
) -> None:
    """Log a model checkpoint as a W&B artifact."""
    if not _run_active:
        return

    artifact = wandb.Artifact(
        f"{algorithm}_checkpoint_step{step}",
        type="model",
        metadata={"step": step, "algorithm": algorithm},
    )
    artifact.add_file(checkpoint_path)
    wandb.log_artifact(artifact)
    logger.info(f"Checkpoint artifact logged: step {step}")


def finish() -> None:
    """Finish the current W&B run and close the local metrics sink."""
    global _run_active, _local_sink
    if _local_sink is not None:
        _local_sink.close()
        _local_sink = None
        logger.info(f"Local metrics written: {_local_sink_path}")
    if _run_active:
        wandb.finish()
        _run_active = False
        logger.info("W&B run finished.")


In [ ]:
!pip install -r requirements.txt
!python main.py --algorithm sgrpo --steps 50 --no_wandb --device cuda